# Bovine Sperm Quality Analysis Pipeline

**Full prototype pipeline for bovine sperm quality analysis from microscopy data, videos, and metadata.**

This notebook discovers public data online, downloads and organizes datasets, audits quality, engineers features, trains baseline and improved models, and outputs predictions at cell, track, and sample levels.

---

## Section 0 — Setup and Configuration

Imports, paths, seeds, config, logging, reproducibility helpers, and optional package installation.

In [ ]:
# ── Optional package installation ──────────────────────────────────────────────
# Uncomment lines below if packages are missing in your environment.
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_optional = [
    "opencv-python-headless",
    "xgboost",
    "ultralytics",        # YOLOv8 for detection
    "roboflow",           # Roboflow dataset download
    "albumentations",     # Image augmentation
    "pyarrow",            # Parquet support
    "openpyxl",           # Excel support
    "shapely",            # Geometry ops
]

for _pkg in _optional:
    try:
        __import__(_pkg.replace("-", "_").split("[")[0])
    except ImportError:
        print(f"Installing {_pkg}...")
        _install(_pkg)

print("All optional packages ready.")

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os, sys, json, hashlib, shutil, glob, logging, warnings, datetime, time
import random, pathlib, io, csv, re, textwrap, collections, itertools
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple, Any, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
import cv2
import yaml
import requests
from tqdm.auto import tqdm
import scipy.stats as stats
import scipy.ndimage as ndi
from scipy.spatial.distance import cdist

from sklearn.model_selection import (
    GroupShuffleSplit, StratifiedGroupKFold, train_test_split, cross_val_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, mean_squared_error,
    mean_absolute_error, r2_score, roc_auc_score, average_precision_score,
    balanced_accuracy_score, ConfusionMatrixDisplay
)
try:
    from sklearn.calibration import calibration_curve
except ImportError:
    calibration_curve = None

from sklearn.linear_model import LogisticRegression, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.pipeline import Pipeline

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

print(f"Python {sys.version}")
print(f"NumPy {np.__version__}, Pandas {pd.__version__}, Scikit-learn {__import__('sklearn').__version__}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def seed_everything(seed=SEED):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
    except ImportError:
        pass

seed_everything()

In [ ]:
# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("/Users/balintmaroti/Documents/bull_sperm")

@dataclass
class ProjectPaths:
    root:            Path = PROJECT_ROOT
    data:            Path = PROJECT_ROOT / "data"
    raw:             Path = PROJECT_ROOT / "data" / "raw"
    raw_external:    Path = PROJECT_ROOT / "data" / "raw" / "external"
    raw_bovine:      Path = PROJECT_ROOT / "data" / "raw" / "bovine"
    raw_human:       Path = PROJECT_ROOT / "data" / "raw" / "human_transfer"
    raw_tabular:     Path = PROJECT_ROOT / "data" / "raw" / "tabular"
    raw_code_refs:   Path = PROJECT_ROOT / "data" / "raw" / "code_refs"
    interim:         Path = PROJECT_ROOT / "data" / "interim"
    processed:       Path = PROJECT_ROOT / "data" / "processed"
    manifests:       Path = PROJECT_ROOT / "data" / "manifests"
    notebooks:       Path = PROJECT_ROOT / "notebooks"
    outputs:         Path = PROJECT_ROOT / "outputs"
    models:          Path = PROJECT_ROOT / "models"
    reports:         Path = PROJECT_ROOT / "reports"
    configs:         Path = PROJECT_ROOT / "configs"

    def ensure_all(self):
        """Create all directories if they don't exist."""
        for f in self.__dataclass_fields__:
            getattr(self, f).mkdir(parents=True, exist_ok=True)
        print(f"All project directories ensured under {self.root}")

PATHS = ProjectPaths()
PATHS.ensure_all()

In [ ]:
# ── Central configuration ─────────────────────────────────────────────────────
CONFIG = {
    "seed": SEED,
    "project_name": "bovine_sperm_quality_v1",
    
    # Data split config
    "split_ratios": {"train": 0.7, "val": 0.15, "test": 0.15},
    "split_unit": "auto",  # Will be determined: bull > ejaculate > video > file
    
    # Image preprocessing
    "image_target_size": (640, 640),
    "image_norm_method": "imagenet",  # or "minmax", "zscore"
    
    # Detection
    "detection_conf_threshold": 0.25,
    "detection_iou_threshold": 0.45,
    
    # Morphology classification
    "morph_classes": ["normal", "abnormal_head", "abnormal_midpiece", 
                      "abnormal_tail", "cytoplasmic_droplet", "other_abnormal", "uncertain"],
    
    # Motion classification
    "motion_classes": ["immotile", "motile_non_progressive", "motile_progressive", "uncertain"],
    
    # Sample quality
    "quality_classes": ["pass", "review", "reject"],
    
    # Label confidence tiers
    "confidence_tiers": ["gold", "silver", "bronze", "synthetic", "unknown"],
    
    # Training
    "batch_size": 32,
    "n_epochs_cnn": 30,
    "learning_rate": 1e-3,
    "early_stopping_patience": 5,
    
    # Experiment tracking
    "experiment_version": "v0.1",
}

print("Configuration loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# ── Logging setup ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("bovine_sperm")
logger.setLevel(logging.INFO)

# ── Experiment log ────────────────────────────────────────────────────────────
EXPERIMENT_LOG = []

def log_experiment(stage: str, model_name: str, data_desc: str, labels: str,
                   split_info: str, metrics: dict, notes: str = "", failures: str = "",
                   next_steps: str = ""):
    """Log an experiment result to the persistent experiment log."""
    entry = {
        "timestamp": datetime.datetime.now().isoformat(),
        "stage": stage,
        "model": model_name,
        "data": data_desc,
        "labels": labels,
        "split": split_info,
        "metrics": metrics,
        "notes": notes,
        "failures": failures,
        "next_steps": next_steps,
    }
    EXPERIMENT_LOG.append(entry)
    logger.info(f"[EXPERIMENT] {stage}/{model_name}: {metrics}")
    return entry

def show_experiment_log():
    """Display the experiment log as a DataFrame."""
    if not EXPERIMENT_LOG:
        print("No experiments logged yet.")
        return pd.DataFrame()
    df = pd.DataFrame(EXPERIMENT_LOG)
    display(df[["stage", "model", "data", "labels", "split", "metrics", "notes"]])
    return df

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def file_hash(filepath, algo="sha256"):
    """Compute hash of a file for integrity checking."""
    h = hashlib.new(algo)
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def download_file(url, dest_path, chunk_size=8192, timeout=120):
    """Download a file with progress bar and return the path."""
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    resp = requests.get(url, stream=True, timeout=timeout)
    resp.raise_for_status()
    total = int(resp.headers.get("content-length", 0))
    with open(dest_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest_path.name) as pbar:
        for chunk in resp.iter_content(chunk_size=chunk_size):
            f.write(chunk)
            pbar.update(len(chunk))
    return dest_path

def load_manifest(path):
    """Load a CSV manifest file as a DataFrame."""
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    logger.warning(f"Manifest not found: {path}")
    return pd.DataFrame()

def save_manifest(df, path):
    """Save a DataFrame as a CSV manifest."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    logger.info(f"Saved manifest: {path} ({len(df)} rows)")

def validate_schema(df, required_cols, name=""):
    """Check that a DataFrame has the required columns."""
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"Schema validation failed for {name}: missing columns {missing}")
    logger.info(f"Schema OK for {name}: {len(df)} rows, {len(df.columns)} cols")

def validate_split_exclusivity(df, group_col, split_col):
    """Ensure no group appears in multiple splits."""
    grouped = df.groupby(group_col)[split_col].nunique()
    violations = grouped[grouped > 1]
    if len(violations) > 0:
        raise ValueError(f"Split leakage! Groups in multiple splits: {violations.index.tolist()[:10]}")
    logger.info(f"Split exclusivity OK: {df[group_col].nunique()} groups across {df[split_col].nunique()} splits")

def create_splits(df, group_col, ratios=None, seed=SEED):
    """Create train/val/test splits by group, preventing leakage."""
    ratios = ratios or CONFIG["split_ratios"]
    groups = df[group_col].unique()
    np.random.seed(seed)
    np.random.shuffle(groups)
    n = len(groups)
    n_train = int(n * ratios["train"])
    n_val = int(n * ratios["val"])
    train_groups = set(groups[:n_train])
    val_groups = set(groups[n_train:n_train + n_val])
    test_groups = set(groups[n_train + n_val:])
    df = df.copy()
    df["split"] = df[group_col].map(
        lambda g: "train" if g in train_groups else ("val" if g in val_groups else "test")
    )
    logger.info(f"Split by {group_col}: train={len(train_groups)}, val={len(val_groups)}, test={len(test_groups)} groups")
    return df

def infer_split_unit(df):
    """Infer the best split unit from available ID columns (bull > ejaculate > video > file)."""
    for col in ["bull_id", "ejaculate_id", "sample_id", "video_id", "file_id", "source_id"]:
        if col in df.columns and df[col].nunique() >= 3:
            logger.info(f"Inferred split unit: {col} ({df[col].nunique()} unique values)")
            return col
    logger.warning("No suitable split unit found; defaulting to index-based split")
    return None

# ── Plotting helpers ──────────────────────────────────────────────────────────

def plot_class_distribution(series, title="Class Distribution", ax=None, top_n=None):
    """Bar plot of class counts."""
    counts = series.value_counts()
    if top_n:
        counts = counts.head(top_n)
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    counts.plot.bar(ax=ax, color=sns.color_palette("muted", len(counts)))
    ax.set_title(title)
    ax.set_ylabel("Count")
    for i, v in enumerate(counts):
        ax.text(i, v + max(counts) * 0.01, str(v), ha="center", fontsize=9)
    plt.tight_layout()
    return ax

def plot_image_grid(images, titles=None, ncols=5, figsize=(15, 8), cmap=None):
    """Display a grid of images."""
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).flatten()
    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(images[i], cmap=cmap)
            if titles:
                ax.set_title(titles[i], fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    return fig

def plot_confusion(y_true, y_pred, labels, title="Confusion Matrix"):
    """Plot a labeled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.tight_layout()
    return fig

# ── Checkpoint helpers ────────────────────────────────────────────────────────

def save_checkpoint(obj, name, path=None):
    """Save a Python object (model, dict, etc.) to disk."""
    import pickle
    path = path or PATHS.models / f"{name}.pkl"
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    logger.info(f"Checkpoint saved: {path}")

def load_checkpoint(name, path=None):
    """Load a checkpoint from disk."""
    import pickle
    path = path or PATHS.models / f"{name}.pkl"
    if not Path(path).exists():
        logger.warning(f"Checkpoint not found: {path}")
        return None
    with open(path, "rb") as f:
        obj = pickle.load(f)
    logger.info(f"Checkpoint loaded: {path}")
    return obj

# ── Metric helpers ────────────────────────────────────────────────────────────

def classification_metrics(y_true, y_pred, y_prob=None, average="macro"):
    """Compute standard classification metrics."""
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        f"f1_{average}": f1_score(y_true, y_pred, average=average, zero_division=0),
        f"precision_{average}": precision_score(y_true, y_pred, average=average, zero_division=0),
        f"recall_{average}": recall_score(y_true, y_pred, average=average, zero_division=0),
    }
    if y_prob is not None and len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
        metrics["avg_precision"] = average_precision_score(y_true, y_prob)
    return metrics

def regression_metrics(y_true, y_pred):
    """Compute standard regression metrics."""
    return {
        "mse": mean_squared_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }

print("All helpers loaded successfully.")

---

## Section 1 — Project Framing

**Objectives:** Build a rigorous, extensible prototype for bovine sperm quality analysis using publicly available data.

**Target hierarchy (multi-level prediction):**
- **Cell-level:** sperm vs non-sperm, normal vs abnormal morphology, defect subtype
- **Track-level:** motile vs immotile, progressive vs non-progressive, kinematic descriptors
- **Sample-level:** concentration, total/progressive motility, morphology quality, pass/review/reject, fertility proxy

**Risks:**
- Public bovine sperm data is sparse; most datasets are small or human-species
- Label quality and consistency across sources will vary
- No pixel-to-micron calibration may be available
- Leakage risk from frame/crop-level splits

**Assumptions:**
- This is a prototype, not a production-grade replacement for CASA labs
- Transfer learning from human sperm data is useful but NOT valid for final bovine evaluation
- Synthetic data augments real data but cannot substitute for real holdout testing

**Success criteria:**
1. At least one bovine dataset downloaded and audited
2. Canonical label schema defined
3. At least one honest baseline model trained per level
4. Predictions produced on a real holdout set
5. Limitations and next steps clearly documented

---

## Section 2 — Online Data Discovery

**Purpose:** Programmatically discover and catalog all public data sources relevant to bovine sperm quality analysis.

**Inputs:** Web search queries, known repository URLs.

**Outputs:** `data_sources_catalog.csv`, ranked source table.

In [ ]:
# ── Data Sources Catalog ──────────────────────────────────────────────────────
# Based on systematic search across Roboflow, Figshare, Kaggle, Zenodo,
# Mendeley, GitHub, and Google Scholar (25+ resources identified).

data_sources = [
    # ── Bovine-first sources (HIGHEST PRIORITY) ──────────────────────────────
    {
        "source_name": "Roboflow — Bovine Sperm Cells Test",
        "url": "https://universe.roboflow.com/boarspermmorphology/bovine-sperm-cells-test",
        "species": "bovine",
        "modality": "images + YOLO annotations",
        "label_types": "9 classes: normal, agglutination, coiled-tail, dirt-particles, distal-droplet, folded-tail, loose-head, loose-tail, proximal-droplet",
        "file_format": "YOLO/COCO/VOC export",
        "approx_size": "277 images, 11 dataset versions",
        "access_method": "Roboflow API (free account)",
        "license": "CC BY 4.0",
        "usefulness": "production_candidate",
        "notes": "BEST bovine source. 9 morphology classes mapping to standard defects. Same user has boar dataset.",
    },
    {
        "source_name": "Roboflow — LPDS Sperm Bovine",
        "url": "https://universe.roboflow.com/amostras-bovinas/lpds-sperm-bovine",
        "species": "bovine",
        "modality": "images + annotations",
        "label_types": "3 classes: e_transversal, erro, mitocondria",
        "file_format": "YOLO/COCO export",
        "approx_size": "99 images",
        "access_method": "Roboflow API",
        "license": "CC BY 4.0",
        "usefulness": "prototype_candidate",
        "notes": "Small. Portuguese labels (Brazilian origin). Structural defect focus.",
    },
    {
        "source_name": "Figshare — CASA Indices Cloned Bulls",
        "url": "https://figshare.com/articles/dataset/CASA_indices_in_frozen-thawed_semen_of_cloned_bulls_and_donors_/12816735",
        "species": "bovine",
        "modality": "tabular (CASA parameters)",
        "label_types": "VCL, VSL, VAP, LIN, STR, WOB, ALH, BCF",
        "file_format": "xlsx",
        "approx_size": "small",
        "access_method": "direct download",
        "license": "journal-linked",
        "usefulness": "reference_only",
        "notes": "CASA kinematic data for cloned bulls. Useful for validating CASA computation.",
    },
    {
        "source_name": "GitHub — YOLO-BullSperm-detection",
        "url": "https://github.com/Gombessa1938/YOLO-BullSperm-detection",
        "species": "bovine",
        "modality": "code (YOLOv5)",
        "label_types": "bull sperm morphology detection",
        "file_format": "Python",
        "approx_size": "WIP",
        "access_method": "git clone",
        "license": "unknown",
        "usefulness": "prototype_candidate",
        "notes": "Directly targets bull sperm with YOLO. May have training configs.",
    },
    # ── Open-source analysis tools ────────────────────────────────────────────
    {
        "source_name": "OpenCASA",
        "url": "https://github.com/calquezar/OpenCASA",
        "species": "multi-species",
        "modality": "software (Java/ImageJ plugin)",
        "label_types": "motility, morphometry, concentration, viability, chemotaxis",
        "file_format": "ImageJ plugin (AVI/JPEG/PNG input)",
        "approx_size": "N/A",
        "access_method": "git clone",
        "license": "GPL (published in PLOS Computational Biology)",
        "usefulness": "production_candidate",
        "notes": "Complete open-source CASA tool. 32 stars. Essential reference implementation.",
    },
    # ── Transfer-learning sources (boar/human) ───────────────────────────────
    {
        "source_name": "Roboflow — Boar Analysis (13 classes)",
        "url": "https://universe.roboflow.com/boarspermmorphology/boar-analisys",
        "species": "boar",
        "modality": "images + annotations",
        "label_types": "13 classes: Normal, asymmetric insertion, Bent-tail, Coiled-tail, Distal-droplet, etc.",
        "file_format": "YOLO/COCO export",
        "approx_size": "395 images",
        "access_method": "Roboflow API",
        "license": "CC BY 4.0",
        "usefulness": "transfer_only",
        "notes": "Richest morphology taxonomy of any dataset found. Same user as bovine. Strong transfer candidate.",
    },
    {
        "source_name": "VISEM-Tracking (Zenodo)",
        "url": "https://zenodo.org/records/7293726",
        "species": "human",
        "modality": "video frames + tracking annotations",
        "label_types": "3 classes: sperm, cluster, small/pinhead",
        "file_format": "ZIP (6.3 GB)",
        "approx_size": "6.3 GB, 1372 downloads",
        "access_method": "direct download",
        "license": "Open access (Scientific Data, Nature 2023)",
        "usefulness": "transfer_only",
        "notes": "Gold-standard human tracking dataset. Best for developing motility algorithms.",
    },
    {
        "source_name": "VISEM Video Dataset (Kaggle)",
        "url": "https://www.kaggle.com/datasets/stevenhicks/visem-video-dataset",
        "species": "human",
        "modality": "video + tabular (semen analysis, fatty acids, hormones)",
        "label_types": "motility grade, concentration, morphology %, volume",
        "file_format": "AVI (640x480, 50fps) + CSV",
        "approx_size": "35.59 GB, 85 videos",
        "access_method": "Kaggle download",
        "license": "CC BY-NC 4.0",
        "usefulness": "transfer_only",
        "notes": "Richest multimodal sperm dataset. Videos + 6 CSV files. 1117 downloads.",
    },
    {
        "source_name": "Kaggle — MHSMA (Human Morphology)",
        "url": "https://www.kaggle.com/datasets/orvile/mhsma-sperm-morphology-analysis-dataset",
        "species": "human",
        "modality": "images (grayscale .npy)",
        "label_types": "4 binary: acrosome, head, tail, vacuole (normal/abnormal)",
        "file_format": ".npy (128x128, 64x64)",
        "approx_size": "32 MB, 1540 images",
        "access_method": "Kaggle download",
        "license": "CC BY-NC 3.0",
        "usefulness": "transfer_only",
        "notes": "Most granular morphological sub-labeling of any dataset. Multi-label.",
    },
    {
        "source_name": "Kaggle — SMIDS (Human Normal/Abnormal/Non-Sperm)",
        "url": "https://www.kaggle.com/datasets/orvile/sperm-morphology-image-data-set-smids",
        "species": "human",
        "modality": "RGB images",
        "label_types": "3 classes: Normal_Sperm (1021), Abnormal_Sperm (1005), Non_Sperm (974)",
        "file_format": "JPEG in folders",
        "approx_size": "100 MB, 3000 images",
        "access_method": "Kaggle download",
        "license": "Other (Mendeley Data origin)",
        "usefulness": "transfer_only",
        "notes": "Smartphone-acquired. Good for transfer learning on normal/abnormal.",
    },
    {
        "source_name": "Roboflow — Sperm Detection Full (Human)",
        "url": "https://universe.roboflow.com/sperm-analysis/sperm-detection-full",
        "species": "human",
        "modality": "images + annotations",
        "label_types": "2 classes: sperm, background",
        "file_format": "YOLO/COCO",
        "approx_size": "1680 images, 7 trained models",
        "access_method": "Roboflow API",
        "license": "CC BY 4.0",
        "usefulness": "transfer_only",
        "notes": "Large human detection dataset. Pre-training only.",
    },
]

# Build catalog DataFrame
sources_catalog = pd.DataFrame(data_sources)
save_manifest(sources_catalog, PATHS.manifests / "data_sources_catalog.csv")

print(f"\n{'='*80}")
print(f"DATA SOURCES CATALOG: {len(sources_catalog)} sources identified")
print(f"{'='*80}")
print(f"\nBy species:")
print(sources_catalog["species"].value_counts().to_string())
print(f"\nBy usefulness:")
print(sources_catalog["usefulness"].value_counts().to_string())
print()
sources_catalog[["source_name", "species", "modality", "usefulness"]]

---

## Section 3 — Data Download and File Inventory

**Purpose:** Download all legally accessible public datasets and organize into standardized folder structure.

**Inputs:** Data sources catalog from Section 2.

**Outputs:** `download_manifest.csv`, `file_inventory.csv`, organized data folders.

In [ ]:
# ── 3A: Download Roboflow Bovine Sperm Datasets ──────────────────────────────
ROBOFLOW_API_KEY = "5GhXqSCLMJPVb0ZfkRMo"
ROBOFLOW_BOVINE_DIR = PATHS.raw_bovine / "roboflow_bovine_sperm"
ROBOFLOW_BOVINE_DIR.mkdir(parents=True, exist_ok=True)

from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

def download_roboflow_project(workspace_id, project_id, dest_parent, version_num=1, fmt="yolov8"):
    """Download a Roboflow project, returning the actual download path."""
    dest_parent = Path(dest_parent).resolve()
    dest_parent.mkdir(parents=True, exist_ok=True)
    try:
        project = rf.workspace(workspace_id).project(project_id)
        orig_dir = os.getcwd()
        os.chdir(str(dest_parent))
        dataset = project.version(version_num).download(fmt)
        os.chdir(orig_dir)
        
        # Resolve the actual path (Roboflow may return relative)
        if hasattr(dataset, 'location'):
            actual_path = Path(dataset.location)
            if not actual_path.is_absolute():
                actual_path = (dest_parent / actual_path).resolve()
            if actual_path.exists():
                logger.info(f"Downloaded {project_id} to {actual_path}")
                return actual_path, True
        
        # Fallback: find newest dir with images in dest_parent
        for d in sorted(dest_parent.iterdir(), key=lambda x: x.stat().st_mtime, reverse=True):
            if d.is_dir() and (list(d.rglob("*.jpg")) or list(d.rglob("*.png"))):
                return d, True
        return None, False
    except Exception as e:
        logger.warning(f"Download failed for {project_id}: {e}")
        try: os.chdir(orig_dir)
        except: pass
        return None, False

# ── Dataset 1: Bovine Sperm Cells Test (8 morphology classes) ────
print("="*80)
print("Downloading: Bovine Sperm Cells Test")
print("="*80)
ds1_path, ds1_success = download_roboflow_project(
    "boarspermmorphology", "bovine-sperm-cells-test", ROBOFLOW_BOVINE_DIR)

# ── Dataset 2: LPDS Sperm Bovine (3 classes) ─────────────────────
print("\n" + "="*80)
print("Downloading: LPDS Sperm Bovine")
print("="*80)
ds2_path, ds2_success = download_roboflow_project(
    "amostras-bovinas", "lpds-sperm-bovine", ROBOFLOW_BOVINE_DIR)

# ── Dataset 3: Boar Analysis for transfer (13 classes) ───────────
print("\n" + "="*80)
print("Downloading: Boar Analysis (transfer learning)")
print("="*80)
ds3_path, ds3_success = download_roboflow_project(
    "boarspermmorphology", "boar-analisys", PATHS.raw_external / "boar_analysis")

# ── Summary ──────────────────────────────────────────────────────
print("\n" + "="*80)
print("ROBOFLOW DOWNLOAD SUMMARY")
print("="*80)
for name, path, ok in [("Bovine Sperm Cells Test", ds1_path, ds1_success),
                        ("LPDS Sperm Bovine", ds2_path, ds2_success),
                        ("Boar Analysis (transfer)", ds3_path, ds3_success)]:
    status = f"OK -> {path}" if ok else "FAILED"
    if ok and path:
        n = len(list(path.rglob("*.jpg"))) + len(list(path.rglob("*.png")))
        status += f" ({n} images)"
    print(f"  {name}: {status}")

real_data_dirs = []
for path, success in [(ds1_path, ds1_success), (ds2_path, ds2_success)]:
    if success and path and path.exists():
        n_imgs = len(list(path.rglob("*.jpg"))) + len(list(path.rglob("*.png")))
        if n_imgs > 0:
            real_data_dirs.append(path)

real_data_exists = len(real_data_dirs) > 0
total_real = sum(len(list(d.rglob("*.jpg"))) + len(list(d.rglob("*.png"))) for d in real_data_dirs)
print(f"\nReal bovine data available: {real_data_exists} ({total_real} images)")

In [ ]:
# ── 3B: Consolidate Real + Synthetic Data ─────────────────────────────────────
# If real Roboflow data was downloaded, parse its annotations and use it.
# Also generate synthetic data as augmentation support (never as final eval).

def parse_yolov8_dataset(dataset_dir, source_name="roboflow"):
    """Parse a YOLOv8 format dataset (train/valid/test splits) into a DataFrame."""
    dataset_dir = Path(dataset_dir)
    all_annotations = []
    
    # Read class names from data.yaml
    yaml_path = dataset_dir / "data.yaml"
    class_names = []
    if yaml_path.exists():
        with open(yaml_path) as f:
            data_yaml = yaml.safe_load(f)
        class_names = data_yaml.get("names", [])
        if isinstance(class_names, dict):
            class_names = [class_names[k] for k in sorted(class_names.keys())]
        logger.info(f"Classes from {yaml_path}: {class_names}")
    
    # Scan train/valid/test splits
    for split_name in ["train", "valid", "test"]:
        img_dir = dataset_dir / split_name / "images"
        lbl_dir = dataset_dir / split_name / "labels"
        if not img_dir.exists():
            continue
        
        for img_path in sorted(img_dir.glob("*")):
            if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png", ".bmp"):
                continue
            
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if not lbl_path.exists():
                continue
            
            with open(lbl_path) as f:
                for j, line in enumerate(f):
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cls_idx = int(parts[0])
                    cx, cy, w, h = [float(x) for x in parts[1:5]]
                    cls_name = class_names[cls_idx] if cls_idx < len(class_names) else f"class_{cls_idx}"
                    
                    all_annotations.append({
                        "class_idx": cls_idx,
                        "class_name": cls_name,
                        "cx": cx, "cy": cy, "w": w, "h": h,
                        "image_file": img_path.name,
                        "image_path": str(img_path),
                        "image_id": img_path.stem,
                        "label_path": str(lbl_path),
                        "original_split": split_name,
                        "source": source_name,
                        "label_confidence": "silver",  # Roboflow community annotations
                    })
    
    df = pd.DataFrame(all_annotations)
    if len(df) > 0:
        logger.info(f"Parsed {source_name}: {len(df)} annotations, {df['image_id'].nunique()} images, {df['class_name'].nunique()} classes")
    return df, class_names

# ── Parse real Roboflow datasets ─────────────────────────────────────────────
all_real_annotations = []
all_class_names = []

# Use actual download paths from cell 3A (ds1_path, ds2_path)
for ds_dir, ds_name in [(ds1_path, "roboflow_bovine_cells"), (ds2_path, "roboflow_lpds_bovine")]:
    if ds_dir.exists():
        df, cls_names = parse_yolov8_dataset(ds_dir, source_name=ds_name)
        if len(df) > 0:
            all_real_annotations.append(df)
            all_class_names.extend(cls_names)
            print(f"\n{ds_name}: {len(df)} annotations from {df['image_id'].nunique()} images")
            print(f"  Classes: {df['class_name'].value_counts().to_dict()}")

if all_real_annotations:
    real_ann_df = pd.concat(all_real_annotations, ignore_index=True)
    all_class_names = list(dict.fromkeys(all_class_names))  # dedupe preserving order
    save_manifest(real_ann_df, PATHS.manifests / "real_annotations.csv")
    print(f"\nTotal real annotations: {len(real_ann_df)} from {real_ann_df['image_id'].nunique()} images")
    print(f"All classes: {all_class_names}")
else:
    real_ann_df = pd.DataFrame()
    print("No real Roboflow data parsed.")

# ── Generate synthetic data as augmentation/fallback ──────────────────────────

def generate_synthetic_sperm_image(img_size=640, n_cells_range=(5, 25), seed=None):
    """Generate a synthetic darkfield microscopy image with sperm-like objects."""
    if seed is not None:
        np.random.seed(seed)
    
    img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
    bg_level = np.random.randint(5, 20)
    img[:] = bg_level
    noise = np.random.normal(0, 3, img.shape).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    n_cells = np.random.randint(*n_cells_range)
    annotations = []
    
    classes = ["normal", "abnormal_head", "abnormal_midpiece", "abnormal_tail", 
               "cytoplasmic_droplet", "other_abnormal"]
    class_weights = [0.55, 0.15, 0.08, 0.10, 0.07, 0.05]
    
    for _ in range(n_cells):
        cx = np.random.randint(40, img_size - 40)
        cy = np.random.randint(40, img_size - 40)
        cls_idx = np.random.choice(len(classes), p=class_weights)
        cls_name = classes[cls_idx]
        
        head_w = np.random.randint(8, 18)
        head_h = np.random.randint(12, 25)
        angle = np.random.randint(0, 360)
        brightness = np.random.randint(140, 230)
        
        if cls_name == "abnormal_head":
            head_w = int(head_w * np.random.uniform(1.3, 1.8))
        elif cls_name == "cytoplasmic_droplet":
            head_h = int(head_h * np.random.uniform(1.2, 1.5))
        
        cv2.ellipse(img, (cx, cy), (head_w, head_h), angle, 0, 360, 
                     (brightness, brightness - 20, brightness - 10), -1)
        
        tail_len = np.random.randint(25, 60)
        if cls_name == "abnormal_tail":
            tail_len = np.random.randint(10, 25)
        
        angle_rad = np.radians(angle)
        pts = []
        for t in range(tail_len):
            curve = np.sin(t * 0.15) * np.random.uniform(2, 6)
            tx = int(cx + (t + head_h) * np.cos(angle_rad) + curve * np.sin(angle_rad))
            ty = int(cy + (t + head_h) * np.sin(angle_rad) - curve * np.cos(angle_rad))
            if 0 <= tx < img_size and 0 <= ty < img_size:
                pts.append([tx, ty])
        
        if len(pts) > 2:
            pts_arr = np.array(pts, dtype=np.int32)
            tail_bright = max(50, brightness - 80)
            cv2.polylines(img, [pts_arr], False, (tail_bright, tail_bright - 10, tail_bright), 1)
        
        all_pts_x = [cx - head_w, cx + head_w] + ([p[0] for p in pts] if pts else [cx])
        all_pts_y = [cy - head_h, cy + head_h] + ([p[1] for p in pts] if pts else [cy])
        x_min, x_max = max(0, min(all_pts_x)), min(img_size, max(all_pts_x))
        y_min, y_max = max(0, min(all_pts_y)), min(img_size, max(all_pts_y))
        
        annotations.append({
            "class_idx": cls_idx, "class_name": cls_name,
            "cx": (x_min + x_max) / 2 / img_size, "cy": (y_min + y_max) / 2 / img_size,
            "w": (x_max - x_min) / img_size, "h": (y_max - y_min) / img_size,
            "x_min": x_min, "y_min": y_min, "x_max": x_max, "y_max": y_max,
        })
    
    n_debris = np.random.randint(0, 8)
    for _ in range(n_debris):
        dx, dy = np.random.randint(10, img_size - 10, 2)
        cv2.circle(img, (int(dx), int(dy)), np.random.randint(2, 8), (np.random.randint(40, 100),)*3, -1)
    
    if np.random.random() > 0.5:
        img = cv2.GaussianBlur(img, (3, 3), 0)
    
    return img, annotations

def generate_synthetic_dataset(n_images=300, img_size=640, base_seed=42):
    """Generate a full synthetic dataset in YOLO format."""
    syn_dir = PATHS.raw_bovine / "synthetic_bovine"
    images_dir = syn_dir / "images"
    labels_dir = syn_dir / "labels"
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    
    class_names = ["normal", "abnormal_head", "abnormal_midpiece", 
                   "abnormal_tail", "cytoplasmic_droplet", "other_abnormal"]
    all_annotations = []
    
    for i in tqdm(range(n_images), desc="Generating synthetic images"):
        img, anns = generate_synthetic_sperm_image(img_size=img_size, seed=base_seed + i)
        img_name = f"syn_{i:05d}.jpg"
        cv2.imwrite(str(images_dir / img_name), img)
        with open(labels_dir / f"syn_{i:05d}.txt", "w") as f:
            for ann in anns:
                f.write(f"{ann['class_idx']} {ann['cx']:.6f} {ann['cy']:.6f} {ann['w']:.6f} {ann['h']:.6f}\n")
        for ann in anns:
            ann.update({"image_file": img_name, "image_id": f"syn_{i:05d}", 
                        "source": "synthetic", "label_confidence": "synthetic"})
            all_annotations.append(ann)
    
    with open(syn_dir / "classes.txt", "w") as f:
        for c in class_names: f.write(c + "\n")
    
    ann_df = pd.DataFrame(all_annotations)
    save_manifest(ann_df, PATHS.manifests / "synthetic_annotations.csv")
    logger.info(f"Generated {n_images} synthetic images, {len(all_annotations)} annotations")
    return ann_df

# Always generate synthetic as augmentation support
print("\nGenerating synthetic data as augmentation support...")
synthetic_ann_df = generate_synthetic_dataset(n_images=300)

# ── Decide primary data source ───────────────────────────────────────────────
if len(real_ann_df) > 0:
    primary_ann_df = real_ann_df
    data_source_type = "real"
    print(f"\n>>> PRIMARY DATA: REAL Roboflow ({len(real_ann_df)} annotations) <<<")
    print("    Synthetic data available as augmentation support.")
else:
    primary_ann_df = synthetic_ann_df
    data_source_type = "synthetic"
    print(f"\n>>> PRIMARY DATA: SYNTHETIC ({len(synthetic_ann_df)} annotations) <<<")
    print("    No real data downloaded — using synthetic for pipeline dev.")

In [ ]:
# ── 3C: Generate Synthetic Tabular CASA Data ──────────────────────────────────
# Realistic sample-level CASA-like data for sample-level modeling pipeline.
# Based on published ranges for bovine semen parameters.

def generate_synthetic_casa_data(n_samples=200, n_bulls=30, seed=42):
    """Generate realistic synthetic CASA-like tabular data for bovine semen samples."""
    np.random.seed(seed)
    
    bulls = [f"BULL_{i:03d}" for i in range(n_bulls)]
    breeds = np.random.choice(["Holstein", "Angus", "Simmental", "Hereford", "Charolais"], n_bulls)
    bull_ages = np.random.uniform(1.5, 10, n_bulls)  # years
    
    records = []
    for i in range(n_samples):
        bull_idx = np.random.randint(0, n_bulls)
        bull_id = bulls[bull_idx]
        breed = breeds[bull_idx]
        age = bull_ages[bull_idx] + np.random.normal(0, 0.2)
        
        # Fresh vs thawed
        fresh_thawed = np.random.choice(["fresh", "frozen_thawed"], p=[0.4, 0.6])
        
        # Base quality depends on bull and fresh/thawed
        bull_quality = np.random.normal(0.7, 0.15)
        thaw_penalty = 0.15 if fresh_thawed == "frozen_thawed" else 0.0
        base_q = np.clip(bull_quality - thaw_penalty + np.random.normal(0, 0.05), 0.1, 0.99)
        
        # CASA parameters (realistic ranges for bovine)
        concentration = np.clip(np.random.normal(800, 400), 50, 3000)  # million/mL
        total_motility = np.clip(base_q * 100 + np.random.normal(0, 8), 5, 99)
        progressive_motility = np.clip(total_motility * np.random.uniform(0.4, 0.8) + np.random.normal(0, 5), 0, total_motility)
        morphology_normal = np.clip(base_q * 80 + np.random.normal(0, 10), 5, 95)
        
        # Kinematic parameters (pixel-based, would need calibration)
        vcl = np.clip(np.random.normal(120, 40), 20, 300)  # curvilinear velocity um/s
        vsl = np.clip(vcl * np.random.uniform(0.3, 0.8), 10, 200)  # straight-line velocity
        vap = np.clip((vcl + vsl) / 2 + np.random.normal(0, 10), 15, 250)  # average path velocity
        lin = np.clip(vsl / vcl * 100, 10, 99)  # linearity
        str_val = np.clip(vsl / vap * 100 if vap > 0 else 50, 10, 99)  # straightness
        wob = np.clip(vap / vcl * 100, 10, 99)  # wobble
        alh = np.clip(np.random.normal(5, 2), 1, 15)  # lateral head amplitude
        bcf = np.clip(np.random.normal(25, 8), 5, 50)  # beat cross frequency
        
        # Defect percentages
        head_defects = np.clip(np.random.exponential(5), 0, 40)
        midpiece_defects = np.clip(np.random.exponential(3), 0, 25)
        tail_defects = np.clip(np.random.exponential(4), 0, 30)
        droplets = np.clip(np.random.exponential(2), 0, 20)
        
        # Sample quality class
        if total_motility >= 60 and morphology_normal >= 60 and progressive_motility >= 30:
            quality_class = "pass"
        elif total_motility >= 40 and morphology_normal >= 40:
            quality_class = "review"
        else:
            quality_class = "reject"
        
        # Fertility proxy (noisy, only available for some)
        has_fertility = np.random.random() > 0.6
        fertility_proxy = np.clip(
            base_q * 60 + np.random.normal(0, 12), 10, 90
        ) if has_fertility else np.nan
        
        # Collection metadata
        month = np.random.randint(1, 13)
        season = {1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
                  6: "summer", 7: "summer", 8: "summer", 9: "fall", 10: "fall",
                  11: "fall", 12: "winter"}[month]
        
        records.append({
            "sample_id": f"S_{i:04d}",
            "ejaculate_id": f"EJ_{i:04d}",
            "bull_id": bull_id,
            "breed": breed,
            "bull_age_years": round(age, 1),
            "collection_month": month,
            "season": season,
            "fresh_or_thawed": fresh_thawed,
            "concentration_M_per_mL": round(concentration, 1),
            "total_motility_pct": round(total_motility, 1),
            "progressive_motility_pct": round(progressive_motility, 1),
            "morphology_normal_pct": round(morphology_normal, 1),
            "vcl_um_s": round(vcl, 1),
            "vsl_um_s": round(vsl, 1),
            "vap_um_s": round(vap, 1),
            "linearity_pct": round(lin, 1),
            "straightness_pct": round(str_val, 1),
            "wobble_pct": round(wob, 1),
            "alh_um": round(alh, 1),
            "bcf_hz": round(bcf, 1),
            "head_defects_pct": round(head_defects, 1),
            "midpiece_defects_pct": round(midpiece_defects, 1),
            "tail_defects_pct": round(tail_defects, 1),
            "droplets_pct": round(droplets, 1),
            "quality_class": quality_class,
            "fertility_proxy": round(fertility_proxy, 1) if has_fertility else np.nan,
            "source": "synthetic_casa",
            "label_confidence": "synthetic",
        })
    
    df = pd.DataFrame(records)
    save_manifest(df, PATHS.raw_tabular / "synthetic_casa_data.csv")
    save_manifest(df, PATHS.manifests / "casa_data.csv")
    
    logger.info(f"Generated {len(df)} synthetic CASA records for {n_bulls} bulls")
    logger.info(f"Quality distribution:\n{df['quality_class'].value_counts().to_string()}")
    logger.info(f"Fertility proxy available: {df['fertility_proxy'].notna().sum()}/{len(df)}")
    
    return df

casa_df = generate_synthetic_casa_data(n_samples=200, n_bulls=30)
print(f"\nCASA data shape: {casa_df.shape}")
print(f"\nFirst 5 rows:")
casa_df.head()

In [ ]:
# ── 3D: File Inventory ────────────────────────────────────────────────────────
# Build comprehensive inventory of all downloaded / generated files.

def build_file_inventory(base_dir):
    """Scan a directory recursively and build a file inventory DataFrame."""
    records = []
    base_dir = Path(base_dir)
    for fp in sorted(base_dir.rglob("*")):
        if fp.is_file():
            ext = fp.suffix.lower()
            size = fp.stat().st_size
            modality = "unknown"
            if ext in (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"):
                modality = "image"
            elif ext in (".mp4", ".avi", ".mov", ".mkv"):
                modality = "video"
            elif ext in (".csv", ".xlsx", ".tsv", ".parquet"):
                modality = "tabular"
            elif ext in (".txt", ".xml", ".json", ".yaml", ".yml"):
                modality = "annotation"
            records.append({
                "file_path": str(fp),
                "relative_path": str(fp.relative_to(base_dir)),
                "filename": fp.name,
                "extension": ext,
                "size_bytes": size,
                "modality": modality,
                "parent_dir": fp.parent.name,
            })
    return pd.DataFrame(records)

file_inv = build_file_inventory(PATHS.data)
save_manifest(file_inv, PATHS.manifests / "file_inventory.csv")

print(f"File Inventory: {len(file_inv)} files")
print(f"\nBy modality:")
print(file_inv["modality"].value_counts().to_string())
print(f"\nBy extension:")
print(file_inv["extension"].value_counts().head(10).to_string())
print(f"\nTotal size: {file_inv['size_bytes'].sum() / 1e6:.1f} MB")

---

## Section 4 — Data Audit and Exploratory Analysis

**Purpose:** Inspect all data before any modeling. Check for corruption, class imbalance, duplicates, annotation quality, and determine the correct split unit.

**Inputs:** Generated/downloaded datasets from Section 3.

**Outputs:** `audit_report.md`, `dataset_health_report.csv`, QC visualizations.

In [ ]:
# ── 4A: Image Data Audit ──────────────────────────────────────────────────────

# Determine image/label directories based on data source
if data_source_type == "real" and len(real_data_dirs) > 0:
    # For real data, scan all split directories
    all_image_files = []
    all_label_files = []
    for ds_dir in real_data_dirs:
        for split_name in ["train", "valid", "test"]:
            img_dir = ds_dir / split_name / "images"
            lbl_dir = ds_dir / split_name / "labels"
            if img_dir.exists():
                all_image_files.extend(sorted(img_dir.glob("*.*")))
            if lbl_dir.exists():
                all_label_files.extend(sorted(lbl_dir.glob("*.txt")))
    image_files = [f for f in all_image_files if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
    label_files = all_label_files
    # Set primary dirs for downstream visualization
    syn_images_dir = real_data_dirs[0] / "train" / "images"
    syn_labels_dir = real_data_dirs[0] / "train" / "labels"
else:
    syn_images_dir = PATHS.raw_bovine / "synthetic_bovine" / "images"
    syn_labels_dir = PATHS.raw_bovine / "synthetic_bovine" / "labels"
    image_files = sorted(syn_images_dir.glob("*.jpg")) if syn_images_dir.exists() else []
    label_files = sorted(syn_labels_dir.glob("*.txt")) if syn_labels_dir.exists() else []

print(f"Data source: {data_source_type.upper()}")
print(f"Image files found: {len(image_files)}")
print(f"Label files found: {len(label_files)}")

# Check for matching image-label pairs
image_stems = {f.stem for f in image_files}
label_stems = {f.stem for f in label_files}
missing_labels = image_stems - label_stems
missing_images = label_stems - image_stems
print(f"Images without labels: {len(missing_labels)}")
print(f"Labels without images: {len(missing_images)}")

# Inspect image dimensions
dims = []
corrupted = []
sample_size = min(100, len(image_files))
for img_path in tqdm(image_files[:sample_size], desc="Auditing images"):
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            corrupted.append(img_path.name)
            continue
        h, w, c = img.shape
        dims.append({"filename": img_path.name, "width": w, "height": h, "channels": c,
                      "size_kb": img_path.stat().st_size / 1024})
    except Exception as e:
        corrupted.append(img_path.name)

dims_df = pd.DataFrame(dims)
print(f"\nCorrupted images: {len(corrupted)}")
if len(dims_df) > 0:
    print(f"\nImage dimensions summary:")
    print(dims_df[["width", "height", "channels", "size_kb"]].describe().round(1))

In [ ]:
# ── 4B: Annotation Class Distribution ─────────────────────────────────────────

ann_df = primary_ann_df.copy()

if len(ann_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Class distribution
    plot_class_distribution(ann_df["class_name"], title=f"Morphology Class Distribution ({data_source_type})", ax=axes[0])
    
    # Objects per image
    objs_per_img = ann_df.groupby("image_id").size()
    axes[1].hist(objs_per_img, bins=20, color="steelblue", edgecolor="white")
    axes[1].set_title(f"Objects per Image (mean={objs_per_img.mean():.1f})")
    axes[1].set_xlabel("Number of objects")
    axes[1].set_ylabel("Frequency")
    
    plt.tight_layout()
    plt.savefig(PATHS.reports / "class_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    print(f"\nData source: {data_source_type.upper()}")
    print(f"Total annotations: {len(ann_df)}")
    print(f"Unique images: {ann_df['image_id'].nunique()}")
    print(f"Classes: {ann_df['class_name'].nunique()}")
    print(f"Objects per image: mean={objs_per_img.mean():.1f}, min={objs_per_img.min()}, max={objs_per_img.max()}")
    print(f"\nClass counts:")
    print(ann_df["class_name"].value_counts().to_string())
else:
    print("No annotations found.")

In [ ]:
# ── 4C: Tabular (CASA) Data Audit ─────────────────────────────────────────────

print("="*80)
print("CASA DATA AUDIT")
print("="*80)

print(f"\nShape: {casa_df.shape}")
print(f"\nMissing values:")
missing = casa_df.isnull().sum()
print(missing[missing > 0].to_string())

print(f"\nNumeric summary:")
numeric_cols = casa_df.select_dtypes(include=[np.number]).columns.tolist()
display(casa_df[numeric_cols].describe().round(2))

# Visualize key distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, col in zip(axes.flatten(), 
                    ["total_motility_pct", "progressive_motility_pct", "morphology_normal_pct",
                     "concentration_M_per_mL", "vcl_um_s", "fertility_proxy"]):
    if col in casa_df.columns:
        casa_df[col].dropna().hist(bins=25, ax=ax, color="steelblue", edgecolor="white")
        ax.set_title(col, fontsize=10)
        ax.axvline(casa_df[col].median(), color="red", linestyle="--", alpha=0.7)

plt.suptitle("CASA Parameter Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PATHS.reports / "casa_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# Quality class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_class_distribution(casa_df["quality_class"], "Sample Quality Class", ax=axes[0])
plot_class_distribution(casa_df["breed"], "Breed Distribution", ax=axes[1])
plt.tight_layout()
plt.show()

# Bull-level summary
bull_summary = casa_df.groupby("bull_id").agg(
    n_samples=("sample_id", "count"),
    mean_motility=("total_motility_pct", "mean"),
    mean_morphology=("morphology_normal_pct", "mean"),
).reset_index().sort_values("n_samples", ascending=False)

print(f"\nBull-level summary ({len(bull_summary)} bulls):")
print(f"  Samples per bull: mean={bull_summary['n_samples'].mean():.1f}, min={bull_summary['n_samples'].min()}, max={bull_summary['n_samples'].max()}")

# ── Determine split unit ──────────────────────────────────────────────────────
split_unit = infer_split_unit(casa_df)
CONFIG["split_unit"] = split_unit or "bull_id"
print(f"\n>>> SPLIT UNIT DETERMINED: {CONFIG['split_unit']} <<<")
print("    All downstream splits will use this unit to prevent leakage.")

In [ ]:
# ── 4D: Sample Image Grid with Annotation Overlay ────────────────────────────

def visualize_annotated_samples(ann_df_sample, n_samples=10):
    """Display sample images with bounding box overlays from annotation DataFrame."""
    unique_classes = sorted(ann_df_sample["class_name"].unique())
    colors = [(0,255,0), (255,0,0), (255,165,0), (0,255,255), (255,0,255), 
              (128,128,0), (128,128,128), (0,128,255), (255,128,0)]
    cls_to_color = {c: colors[i % len(colors)] for i, c in enumerate(unique_classes)}
    
    sample_images = ann_df_sample["image_id"].unique()[:n_samples]
    n = min(n_samples, len(sample_images))
    ncols = min(5, n)
    nrows = max(1, (n + ncols - 1) // ncols)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    if n == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    
    for idx, img_id in enumerate(sample_images):
        ax = axes[idx]
        img_anns = ann_df_sample[ann_df_sample["image_id"] == img_id]
        
        # Get image path
        if "image_path" in img_anns.columns and pd.notna(img_anns.iloc[0].get("image_path")):
            img_path = Path(img_anns.iloc[0]["image_path"])
        else:
            img_path = PATHS.raw_bovine / "synthetic_bovine" / "images" / img_anns.iloc[0]["image_file"]
        
        if not img_path.exists():
            ax.set_title(f"{img_id} (missing)", fontsize=8)
            ax.axis("off")
            continue
        
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        for _, ann in img_anns.iterrows():
            cx, cy, bw, bh = ann["cx"], ann["cy"], ann["w"], ann["h"]
            x1 = int((cx - bw/2) * w)
            y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w)
            y2 = int((cy + bh/2) * h)
            color = cls_to_color.get(ann["class_name"], (255, 255, 255))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            label_short = ann["class_name"][:8]
            cv2.putText(img, label_short, (x1, max(y1-5, 10)), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
        
        ax.imshow(img)
        ax.set_title(f"{img_id}", fontsize=7)
        ax.axis("off")
    
    for i in range(n, len(axes)):
        axes[i].axis("off")
    
    plt.suptitle(f"Sample Images with Annotations ({data_source_type.upper()} data)", 
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PATHS.reports / "sample_images_annotated.png", dpi=150, bbox_inches="tight")
    plt.show()

visualize_annotated_samples(primary_ann_df, n_samples=10)

# ── Audit Summary ─────────────────────────────────────────────────────────────
audit_report = {
    "n_images": len(image_files),
    "n_labels": len(label_files),
    "n_corrupted": len(corrupted),
    "n_missing_labels": len(missing_labels),
    "n_annotations": len(primary_ann_df),
    "n_classes": primary_ann_df["class_name"].nunique(),
    "classes": sorted(primary_ann_df["class_name"].unique().tolist()),
    "n_casa_samples": len(casa_df),
    "n_bulls": casa_df["bull_id"].nunique(),
    "split_unit": CONFIG["split_unit"],
    "data_source": data_source_type,
    "audit_status": "PASSED",
}

print("\n" + "="*80)
print("AUDIT SUMMARY")
print("="*80)
for k, v in audit_report.items():
    print(f"  {k}: {v}")

---

## Section 5 — Canonical Schema and Label Mapping

**Purpose:** Define a unified data model, label taxonomy, and metadata schema. Map all source labels into canonical form.

**Inputs:** Source-specific labels from downloaded data.

**Outputs:** `label_schema.yaml`, `label_mapping.csv`, canonical data model.

In [ ]:
# ── 5A: Label Taxonomy ────────────────────────────────────────────────────────

label_schema = {
    "morphology_labels": {
        "hierarchy": "cell-level",
        "classes": {
            "normal": {"code": 0, "description": "Normal morphology, no visible defects"},
            "abnormal_head": {"code": 1, "description": "Head defects: pyriform, round, macro, micro, detached"},
            "abnormal_midpiece": {"code": 2, "description": "Midpiece defects: bent, thickened, thin"},
            "abnormal_tail": {"code": 3, "description": "Tail defects: coiled, short, bent, absent"},
            "cytoplasmic_droplet": {"code": 4, "description": "Proximal or distal cytoplasmic droplet"},
            "other_abnormal": {"code": 5, "description": "Other abnormalities not classified above"},
            "uncertain": {"code": 6, "description": "Cannot determine class from available data"},
        },
    },
    "motion_labels": {
        "hierarchy": "track-level",
        "classes": {
            "immotile": {"code": 0, "description": "No meaningful displacement"},
            "motile_non_progressive": {"code": 1, "description": "Moving but no forward progress"},
            "motile_progressive": {"code": 2, "description": "Moving with sustained forward progress"},
            "uncertain": {"code": 3, "description": "Cannot determine motility state"},
        },
    },
    "sample_labels": {
        "hierarchy": "sample-level",
        "numeric_targets": [
            "concentration_M_per_mL",
            "total_motility_pct",
            "progressive_motility_pct",
            "morphology_normal_pct",
            "fertility_proxy",
        ],
        "class_targets": {
            "quality_class": {"classes": ["pass", "review", "reject"]},
            "post_thaw_flag": {"classes": ["fresh", "frozen_thawed"]},
        },
    },
    "confidence_tiers": {
        "gold": "Expert human annotation, high confidence",
        "silver": "Curated or consensus annotation",
        "bronze": "Derived or weak label",
        "synthetic": "Simulator-generated",
        "unknown": "Provenance unclear",
    },
}

# Save label schema
with open(PATHS.configs / "label_schema.yaml", "w") as f:
    yaml.dump(label_schema, f, default_flow_style=False, sort_keys=False)
logger.info(f"Label schema saved to {PATHS.configs / 'label_schema.yaml'}")

# ── 5B: Label Mapping Table ──────────────────────────────────────────────────
# Maps source-specific labels to canonical labels

label_mapping = [
    # Morphology mappings
    {"source": "synthetic_bovine", "source_label": "normal", "canonical_label": "normal", "level": "morphology"},
    {"source": "synthetic_bovine", "source_label": "abnormal_head", "canonical_label": "abnormal_head", "level": "morphology"},
    {"source": "synthetic_bovine", "source_label": "abnormal_midpiece", "canonical_label": "abnormal_midpiece", "level": "morphology"},
    {"source": "synthetic_bovine", "source_label": "abnormal_tail", "canonical_label": "abnormal_tail", "level": "morphology"},
    {"source": "synthetic_bovine", "source_label": "cytoplasmic_droplet", "canonical_label": "cytoplasmic_droplet", "level": "morphology"},
    {"source": "synthetic_bovine", "source_label": "other_abnormal", "canonical_label": "other_abnormal", "level": "morphology"},
    # Roboflow potential mappings (for when real data is available)
    {"source": "roboflow_bovine", "source_label": "Head defect", "canonical_label": "abnormal_head", "level": "morphology"},
    {"source": "roboflow_bovine", "source_label": "Tail defect", "canonical_label": "abnormal_tail", "level": "morphology"},
    {"source": "roboflow_bovine", "source_label": "Normal", "canonical_label": "normal", "level": "morphology"},
    # Motion mappings
    {"source": "derived_tracking", "source_label": "static", "canonical_label": "immotile", "level": "motion"},
    {"source": "derived_tracking", "source_label": "non_progressive", "canonical_label": "motile_non_progressive", "level": "motion"},
    {"source": "derived_tracking", "source_label": "progressive", "canonical_label": "motile_progressive", "level": "motion"},
]

label_mapping_df = pd.DataFrame(label_mapping)
save_manifest(label_mapping_df, PATHS.configs / "label_mapping.csv")

print("Label Schema:")
for level, info in label_schema.items():
    if isinstance(info, dict) and "classes" in info:
        print(f"\n  {level} ({info.get('hierarchy', 'N/A')}):")
        for cls, details in info["classes"].items():
            if isinstance(details, dict):
                print(f"    [{details.get('code', '?')}] {cls}: {details.get('description', '')}")

print(f"\nLabel mapping table: {len(label_mapping_df)} entries")

# ── 5C: Canonical ID Schema ──────────────────────────────────────────────────
CANONICAL_IDS = ["source_id", "dataset_id", "file_id", "image_id", "video_id",
                 "frame_id", "cell_id", "track_id", "sample_id", "ejaculate_id", "bull_id"]
CANONICAL_METADATA = ["species", "modality", "split_group", "label_source", 
                      "label_confidence", "device_id", "operator_id", "collection_date",
                      "fresh_or_thawed", "annotation_format", "source_url"]

print(f"\nCanonical ID fields: {CANONICAL_IDS}")
print(f"Canonical metadata fields: {CANONICAL_METADATA}")

---

## Section 6 — Data Preparation Pipelines

**Purpose:** Build preprocessing pipelines for images, frame extraction, annotation conversion, crop generation, and split assignment.

**Inputs:** Raw images/labels from Section 3, schema from Section 5, split unit from Section 4.

**Outputs:** Processed image folders, crop datasets, frame manifests, train/val/test split manifests.

In [ ]:
# ── 6A: Image Preprocessing Pipeline ─────────────────────────────────────────

def preprocess_image(img, target_size=None, normalize=True, method="imagenet"):
    """Preprocess a single image for model input."""
    if target_size:
        img = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32)
    if normalize:
        if method == "imagenet":
            mean = np.array([0.485, 0.456, 0.406]) * 255
            std = np.array([0.229, 0.224, 0.225]) * 255
            img = (img - mean) / std
        elif method == "minmax":
            img = img / 255.0
        elif method == "zscore":
            img = (img - img.mean()) / (img.std() + 1e-8)
    return img

# ── 6B: Crop Extraction from Annotations ─────────────────────────────────────
# Works with both real (Roboflow) and synthetic data.

def extract_crops_from_annotations(ann_df, output_dir, padding_ratio=0.15,
                                    target_crop_size=(64, 64)):
    """Extract cell crops from annotated images. Handles both real and synthetic paths."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Get unique class names and create subdirs
    class_names = sorted(ann_df["class_name"].unique())
    for cls in class_names:
        (output_dir / cls).mkdir(parents=True, exist_ok=True)
    
    crops_manifest = []
    images_processed = set()
    
    # Group annotations by image
    for image_id, group in tqdm(ann_df.groupby("image_id"), desc="Extracting crops"):
        # Determine image path
        if "image_path" in group.columns and pd.notna(group.iloc[0].get("image_path")):
            img_path = Path(group.iloc[0]["image_path"])
        else:
            # Synthetic data — look in synthetic dir
            img_path = PATHS.raw_bovine / "synthetic_bovine" / "images" / group.iloc[0]["image_file"]
        
        if not img_path.exists():
            continue
        
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        
        for j, (_, ann) in enumerate(group.iterrows()):
            cx, cy, bw, bh = ann["cx"], ann["cy"], ann["w"], ann["h"]
            
            pad_w = bw * padding_ratio
            pad_h = bh * padding_ratio
            x1 = max(0, int((cx - bw/2 - pad_w) * w))
            y1 = max(0, int((cy - bh/2 - pad_h) * h))
            x2 = min(w, int((cx + bw/2 + pad_w) * w))
            y2 = min(h, int((cy + bh/2 + pad_h) * h))
            
            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            
            crop_resized = cv2.resize(crop, target_crop_size, interpolation=cv2.INTER_LINEAR)
            
            cls_name = ann["class_name"]
            crop_name = f"{image_id}_crop{j:03d}.jpg"
            crop_path = output_dir / cls_name / crop_name
            cv2.imwrite(str(crop_path), crop_resized)
            
            crops_manifest.append({
                "cell_id": f"{image_id}_c{j:03d}",
                "image_file": ann.get("image_file", ""),
                "image_id": image_id,
                "crop_file": str(crop_path.relative_to(output_dir)),
                "class_idx": int(ann["class_idx"]),
                "class_name": cls_name,
                "bbox_cx": cx, "bbox_cy": cy, "bbox_w": bw, "bbox_h": bh,
                "crop_w": target_crop_size[0],
                "crop_h": target_crop_size[1],
                "source": ann.get("source", "unknown"),
                "label_confidence": ann.get("label_confidence", "unknown"),
            })
    
    return pd.DataFrame(crops_manifest)

# Extract crops from primary data
CROPS_DIR = PATHS.processed / "morphology_crops"
crops_df = extract_crops_from_annotations(primary_ann_df, CROPS_DIR, target_crop_size=(64, 64))
save_manifest(crops_df, PATHS.manifests / "crops_manifest.csv")

print(f"\nExtracted {len(crops_df)} cell crops from {data_source_type} data")
print(f"Crop class distribution:")
print(crops_df["class_name"].value_counts().to_string())

In [ ]:
# ── 6C: Train/Val/Test Split Assignment ───────────────────────────────────────
# Split images by image_id groups (since we lack bull IDs for image data,
# we split by image to prevent crop leakage from same image).

# For image data: split by image
crops_df["split_group"] = crops_df["image_id"]
unique_images = crops_df["split_group"].unique()
np.random.seed(SEED)
np.random.shuffle(unique_images)

n = len(unique_images)
n_train = int(n * 0.7)
n_val = int(n * 0.15)

train_imgs = set(unique_images[:n_train])
val_imgs = set(unique_images[n_train:n_train + n_val])
test_imgs = set(unique_images[n_train + n_val:])

crops_df["split"] = crops_df["split_group"].map(
    lambda x: "train" if x in train_imgs else ("val" if x in val_imgs else "test")
)

print("Image-level crop splits:")
for sp in ["train", "val", "test"]:
    sub = crops_df[crops_df["split"] == sp]
    print(f"  {sp}: {len(sub)} crops from {sub['image_id'].nunique()} images")

# Validate split exclusivity
validate_split_exclusivity(crops_df, "image_id", "split")

# For CASA data: split by bull_id
casa_df = create_splits(casa_df, "bull_id")

print("\nCASA bull-level splits:")
for sp in ["train", "val", "test"]:
    sub = casa_df[casa_df["split"] == sp]
    print(f"  {sp}: {len(sub)} samples from {sub['bull_id'].nunique()} bulls")

validate_split_exclusivity(casa_df, "bull_id", "split")

# Save split manifests
save_manifest(crops_df, PATHS.manifests / "crops_manifest_with_splits.csv")
save_manifest(casa_df, PATHS.manifests / "casa_data_with_splits.csv")

---

## Section 7 — Baseline Open-Source Analytics Integration

**Purpose:** Establish baseline analytics using transparent, rule-based methods before any ML.

**Inputs:** Processed images, annotations, CASA data.

**Outputs:** Baseline metrics, pseudo-labels, benchmark comparisons.

In [ ]:
# ── 7A: Rule-Based Morphology Baseline ────────────────────────────────────────
# Simple heuristic classifier based on crop shape features.
# This is our "transparent baseline" — no ML, just image statistics.

def extract_basic_shape_features(crop_img):
    """Extract basic shape features from a crop image for rule-based classification."""
    if len(crop_img.shape) == 3:
        gray = cv2.cvtColor(crop_img, cv2.COLOR_BGR2GRAY)
    else:
        gray = crop_img
    
    # Threshold to get binary mask
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Find contours
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    features = {}
    if len(contours) > 0:
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        
        features["raw_area"] = area
        features["raw_perimeter"] = perimeter
        features["raw_circularity"] = 4 * np.pi * area / (perimeter ** 2 + 1e-8)
        
        if len(cnt) >= 5:
            ellipse = cv2.fitEllipse(cnt)
            (ex, ey), (ma, MA), angle = ellipse
            features["raw_aspect_ratio"] = max(ma, MA) / (min(ma, MA) + 1e-8)
            features["raw_eccentricity"] = np.sqrt(1 - (min(ma, MA) / (max(ma, MA) + 1e-8))**2)
        else:
            x, y, w, h = cv2.boundingRect(cnt)
            features["raw_aspect_ratio"] = max(w, h) / (min(w, h) + 1e-8)
            features["raw_eccentricity"] = 0.5
        
        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)
        features["raw_solidity"] = area / (hull_area + 1e-8)
        features["raw_convexity"] = cv2.arcLength(hull, True) / (perimeter + 1e-8)
    else:
        features = {k: 0.0 for k in ["raw_area", "raw_perimeter", "raw_circularity",
                                       "raw_aspect_ratio", "raw_eccentricity", "raw_solidity", "raw_convexity"]}
    
    # Intensity features
    features["raw_mean_intensity"] = float(gray.mean())
    features["raw_std_intensity"] = float(gray.std())
    features["raw_intensity_q25"] = float(np.percentile(gray, 25))
    features["raw_intensity_q75"] = float(np.percentile(gray, 75))
    
    return features

# ── 7B: Rule-Based Motility Classification ───────────────────────────────────
# Based on CASA-like thresholds from published bovine literature

def rule_based_motility_classification(row):
    """Classify motility state using published threshold rules."""
    # Standard CASA thresholds for bovine (approximate)
    vcl = row.get("vcl_um_s", 0)
    vsl = row.get("vsl_um_s", 0)
    lin = row.get("linearity_pct", 0)
    str_val = row.get("straightness_pct", 0)
    
    if vcl < 10:  # Nearly static
        return "immotile"
    elif lin > 50 and str_val > 50 and vsl > 25:
        return "motile_progressive"
    else:
        return "motile_non_progressive"

# Apply rule-based motility to CASA data
casa_df["rule_motility_class"] = casa_df.apply(rule_based_motility_classification, axis=1)

print("Rule-based motility classification (CASA thresholds):")
print(casa_df["rule_motility_class"].value_counts().to_string())

# Rule-based quality classification
def rule_based_quality(row):
    """Simple threshold-based quality classification."""
    if row["total_motility_pct"] >= 60 and row["morphology_normal_pct"] >= 60:
        return "pass"
    elif row["total_motility_pct"] >= 40 and row["morphology_normal_pct"] >= 40:
        return "review"
    else:
        return "reject"

casa_df["rule_quality_class"] = casa_df.apply(rule_based_quality, axis=1)

# Compare rule-based vs actual labels
agreement = (casa_df["rule_quality_class"] == casa_df["quality_class"]).mean()
print(f"\nRule-based vs actual quality agreement: {agreement:.1%}")

log_experiment(
    stage="baseline", model_name="rule_based_quality",
    data_desc="CASA synthetic data (200 samples)",
    labels="quality_class (pass/review/reject)",
    split_info="N/A (rule-based, no training)",
    metrics={"agreement_with_labels": round(agreement, 3)},
    notes="Simple threshold baseline. No ML involved.",
    next_steps="Train ML models to improve upon this baseline."
)

---

## Section 8 — Feature Engineering

**Purpose:** Build morphology features (classical + learned), motion/track features, sample-level aggregation, and metadata/process features.

**Inputs:** Cell crops, annotations, CASA data, metadata.

**Outputs:** `cell_morph_features.parquet`, `sample_morph_features.parquet`, `sample_metadata_features.parquet`.

In [ ]:
# ── 8A: Cell-Level Morphology Features (Classical) ────────────────────────────
# Extract interpretable image features from each cell crop.

def extract_morphology_features_batch(crops_df, crops_base_dir):
    """Extract classical morphology features from all crops."""
    features_list = []
    
    for _, row in tqdm(crops_df.iterrows(), total=len(crops_df), desc="Extracting morphology features"):
        crop_path = Path(crops_base_dir) / row["crop_file"]
        if not crop_path.exists():
            continue
        
        crop = cv2.imread(str(crop_path))
        if crop is None:
            continue
        
        feats = extract_basic_shape_features(crop)
        feats["cell_id"] = row["cell_id"]
        feats["class_name"] = row["class_name"]
        feats["class_idx"] = row["class_idx"]
        feats["image_id"] = row["image_id"]
        
        # Additional texture features
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        
        # Texture: Laplacian variance (edge sharpness)
        feats["raw_laplacian_var"] = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        
        # Texture: local binary pattern approximation via gradient magnitude
        gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        grad_mag = np.sqrt(gx**2 + gy**2)
        feats["raw_gradient_mean"] = float(grad_mag.mean())
        feats["raw_gradient_std"] = float(grad_mag.std())
        
        # Symmetry: compare left/right halves
        mid = gray.shape[1] // 2
        if mid > 0:
            left = gray[:, :mid].astype(float)
            right = np.fliplr(gray[:, -mid:]).astype(float)
            min_w = min(left.shape[1], right.shape[1])
            feats["raw_lr_symmetry"] = float(1.0 - np.mean(np.abs(left[:, :min_w] - right[:, :min_w])) / 255.0)
        else:
            feats["raw_lr_symmetry"] = 0.5
        
        features_list.append(feats)
    
    return pd.DataFrame(features_list)

cell_features_df = extract_morphology_features_batch(crops_df, CROPS_DIR)

# Save cell features
cell_features_df.to_parquet(PATHS.processed / "cell_morph_features.parquet", index=False)
save_manifest(cell_features_df, PATHS.manifests / "cell_morph_features.csv")

print(f"Cell morphology features: {cell_features_df.shape}")
print(f"\nFeature columns: {[c for c in cell_features_df.columns if c.startswith('raw_')]}")
print(f"\nFeature summary:")
feat_cols = [c for c in cell_features_df.columns if c.startswith("raw_")]
display(cell_features_df[feat_cols].describe().round(3))

In [ ]:
# ── 8B: Metadata and Process Features ─────────────────────────────────────────
# Engineer features from CASA tabular data for sample-level modeling.

def engineer_casa_features(df):
    """Create engineered features from CASA data."""
    df = df.copy()
    
    # Seasonal indicators
    df["fe_is_summer"] = (df["season"] == "summer").astype(int)
    df["fe_is_winter"] = (df["season"] == "winter").astype(int)
    
    # Age bins
    df["fe_age_bin"] = pd.cut(df["bull_age_years"], bins=[0, 2, 4, 6, 8, 15], 
                               labels=["young", "prime1", "prime2", "mature", "senior"])
    
    # Fresh/thawed flag
    df["fe_is_frozen"] = (df["fresh_or_thawed"] == "frozen_thawed").astype(int)
    
    # Derived motility ratios
    df["fe_prog_to_total_ratio"] = df["progressive_motility_pct"] / (df["total_motility_pct"] + 1e-8)
    df["fe_immotile_pct"] = 100 - df["total_motility_pct"]
    
    # Kinematic derived features
    df["fe_vcl_vsl_ratio"] = df["vcl_um_s"] / (df["vsl_um_s"] + 1e-8)
    df["fe_speed_index"] = (df["vcl_um_s"] + df["vsl_um_s"] + df["vap_um_s"]) / 3
    
    # Total defects
    defect_cols = ["head_defects_pct", "midpiece_defects_pct", "tail_defects_pct", "droplets_pct"]
    df["fe_total_defects_pct"] = df[defect_cols].sum(axis=1)
    df["fe_major_defect_type"] = df[defect_cols].idxmax(axis=1).str.replace("_pct", "")
    
    # Interaction: motility x morphology
    df["fe_motility_morph_score"] = df["total_motility_pct"] * df["morphology_normal_pct"] / 100
    
    # Log transforms for skewed features
    for col in ["concentration_M_per_mL", "vcl_um_s"]:
        df[f"fe_log_{col}"] = np.log1p(df[col])
    
    # Group-level features: bull-level means (rolling historical quality)
    bull_means = df.groupby("bull_id").agg(
        agg_bull_mean_motility=("total_motility_pct", "mean"),
        agg_bull_mean_morphology=("morphology_normal_pct", "mean"),
        agg_bull_n_samples=("sample_id", "count"),
    ).reset_index()
    df = df.merge(bull_means, on="bull_id", how="left")
    
    # Deviation from bull mean
    df["fe_motility_vs_bull_mean"] = df["total_motility_pct"] - df["agg_bull_mean_motility"]
    df["fe_morphology_vs_bull_mean"] = df["morphology_normal_pct"] - df["agg_bull_mean_morphology"]
    
    # Missingness indicators
    df["fe_fertility_missing"] = df["fertility_proxy"].isna().astype(int)
    
    return df

casa_df = engineer_casa_features(casa_df)

# Save feature table
casa_df.to_parquet(PATHS.processed / "sample_features.parquet", index=False)
save_manifest(casa_df, PATHS.manifests / "sample_features.csv")

fe_cols = [c for c in casa_df.columns if c.startswith("fe_") or c.startswith("agg_")]
print(f"Engineered {len(fe_cols)} sample-level features:")
for col in fe_cols:
    print(f"  {col}: {casa_df[col].dtype}")

# ── Feature Distributions ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
plot_cols = ["fe_prog_to_total_ratio", "fe_speed_index", "fe_total_defects_pct",
             "fe_motility_morph_score", "fe_motility_vs_bull_mean", "fe_log_concentration_M_per_mL"]

for ax, col in zip(axes.flatten(), plot_cols):
    if col in casa_df.columns:
        casa_df[col].dropna().hist(bins=25, ax=ax, color="teal", edgecolor="white")
        ax.set_title(col, fontsize=9)

plt.suptitle("Engineered Feature Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PATHS.reports / "engineered_features.png", dpi=150, bbox_inches="tight")
plt.show()

---

## Section 9 — Label Construction

**Purpose:** Consolidate all label sources into unified label tables with confidence tiers.

**Inputs:** Annotations, CASA data, rule-based pseudo-labels.

**Outputs:** `labels_long_table`, label confidence assignments.

In [ ]:
# ── 9: Unified Label Table ────────────────────────────────────────────────────

# Cell-level morphology labels
cell_labels = crops_df[["cell_id", "image_id", "class_name", "class_idx"]].copy()
cell_labels.rename(columns={"class_name": "label_value", "class_idx": "label_code"}, inplace=True)
cell_labels["label_name"] = "morphology_class"
cell_labels["label_source_type"] = "synthetic"
cell_labels["label_source_id"] = "synthetic_generator"
cell_labels["label_confidence"] = "synthetic"
cell_labels["level"] = "cell"

# Sample-level labels (from CASA)
sample_label_rows = []
for _, row in casa_df.iterrows():
    for target in ["quality_class", "total_motility_pct", "progressive_motility_pct",
                    "morphology_normal_pct", "concentration_M_per_mL", "fertility_proxy"]:
        if pd.notna(row.get(target)):
            sample_label_rows.append({
                "cell_id": None,
                "image_id": None,
                "sample_id": row["sample_id"],
                "bull_id": row["bull_id"],
                "label_name": target,
                "label_value": str(row[target]),
                "label_code": None,
                "label_source_type": "synthetic",
                "label_source_id": "synthetic_casa_generator",
                "label_confidence": "synthetic",
                "level": "sample",
            })

# Track-level labels (derived from rule-based classification)
track_label_rows = []
for _, row in casa_df.iterrows():
    track_label_rows.append({
        "cell_id": None,
        "image_id": None,
        "sample_id": row["sample_id"],
        "bull_id": row["bull_id"],
        "label_name": "motility_class",
        "label_value": row["rule_motility_class"],
        "label_code": None,
        "label_source_type": "derived",
        "label_source_id": "rule_based_casa_thresholds",
        "label_confidence": "bronze",
        "level": "track",
    })

sample_labels_df = pd.DataFrame(sample_label_rows)
track_labels_df = pd.DataFrame(track_label_rows)

# Combine all labels
labels_long = pd.concat([cell_labels, sample_labels_df, track_labels_df], ignore_index=True)
save_manifest(labels_long, PATHS.manifests / "labels_long_table.csv")

print(f"Unified label table: {len(labels_long)} entries")
print(f"\nBy level:")
print(labels_long["level"].value_counts().to_string())
print(f"\nBy confidence tier:")
print(labels_long["label_confidence"].value_counts().to_string())
print(f"\nBy label name:")
print(labels_long["label_name"].value_counts().to_string())

---

## Section 10 — Modeling: Cell Level

**Purpose:** Train sperm detection and morphology classification models.

**Streams:**
- Stream A: Sperm detection (YOLO-based)
- Stream B: Morphology classification (Logistic Regression -> Random Forest -> CNN)

**Inputs:** Cell crops with features, YOLO-formatted dataset.

**Outputs:** Trained models, metrics, confusion matrices, error analysis.

In [ ]:
# ── 10A: Morphology Classification — Data Preparation ─────────────────────────
# Merge crop features with split assignments for ML.

# Join cell features with split info from crops_df
morph_ml_df = cell_features_df.merge(
    crops_df[["cell_id", "split"]],
    on="cell_id", how="left"
)

# Define feature columns and target
feature_cols = [c for c in morph_ml_df.columns if c.startswith("raw_")]
target_col = "class_name"

# Binary target: normal vs abnormal (simpler first)
morph_ml_df["binary_target"] = (morph_ml_df["class_name"] != "normal").astype(int)

# Filter train/val/test
train_morph = morph_ml_df[morph_ml_df["split"] == "train"].copy()
val_morph = morph_ml_df[morph_ml_df["split"] == "val"].copy()
test_morph = morph_ml_df[morph_ml_df["split"] == "test"].copy()

# Handle any NaN features
for df in [train_morph, val_morph, test_morph]:
    df[feature_cols] = df[feature_cols].fillna(0)

X_train = train_morph[feature_cols].values
y_train_binary = train_morph["binary_target"].values
y_train_multi = train_morph["class_idx"].values

X_val = val_morph[feature_cols].values
y_val_binary = val_morph["binary_target"].values
y_val_multi = val_morph["class_idx"].values

X_test = test_morph[feature_cols].values
y_test_binary = test_morph["binary_target"].values
y_test_multi = test_morph["class_idx"].values

# Scale features
scaler_morph = RobustScaler()
X_train_sc = scaler_morph.fit_transform(X_train)
X_val_sc = scaler_morph.transform(X_val)
X_test_sc = scaler_morph.transform(X_test)

print(f"Morphology ML data:")
print(f"  Train: {len(train_morph)} ({train_morph['binary_target'].mean():.1%} abnormal)")
print(f"  Val:   {len(val_morph)} ({val_morph['binary_target'].mean():.1%} abnormal)")
print(f"  Test:  {len(test_morph)} ({test_morph['binary_target'].mean():.1%} abnormal)")
print(f"  Features: {len(feature_cols)}")

In [ ]:
# ── 10B: Morphology Classification — Binary (Normal vs Abnormal) ──────────────
# Progressive model complexity: LogReg -> RF -> XGBoost

morph_binary_results = {}

# --- Model 1: Logistic Regression ---
lr_morph = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
lr_morph.fit(X_train_sc, y_train_binary)
y_pred_lr = lr_morph.predict(X_test_sc)
y_prob_lr = lr_morph.predict_proba(X_test_sc)[:, 1]

metrics_lr = classification_metrics(y_test_binary, y_pred_lr, y_prob_lr)
morph_binary_results["LogisticRegression"] = metrics_lr
save_checkpoint(lr_morph, "morph_binary_logreg")

print("="*80)
print("BINARY MORPHOLOGY: Logistic Regression")
print("="*80)
print(classification_report(y_test_binary, y_pred_lr, target_names=["normal", "abnormal"]))

log_experiment(
    stage="cell_morphology_binary", model_name="LogisticRegression",
    data_desc=f"Cell crops, {len(X_train)} train, {len(X_test)} test",
    labels="normal vs abnormal (binary)",
    split_info="Image-level split, 70/15/15",
    metrics=metrics_lr,
    notes="Baseline. RobustScaler + class_weight=balanced.",
    next_steps="Try Random Forest and XGBoost."
)

# --- Model 2: Random Forest ---
rf_morph = RandomForestClassifier(n_estimators=200, class_weight="balanced", 
                                   max_depth=10, random_state=SEED, n_jobs=-1)
rf_morph.fit(X_train_sc, y_train_binary)
y_pred_rf = rf_morph.predict(X_test_sc)
y_prob_rf = rf_morph.predict_proba(X_test_sc)[:, 1]

metrics_rf = classification_metrics(y_test_binary, y_pred_rf, y_prob_rf)
morph_binary_results["RandomForest"] = metrics_rf
save_checkpoint(rf_morph, "morph_binary_rf")

print("\n" + "="*80)
print("BINARY MORPHOLOGY: Random Forest")
print("="*80)
print(classification_report(y_test_binary, y_pred_rf, target_names=["normal", "abnormal"]))

log_experiment(
    stage="cell_morphology_binary", model_name="RandomForest",
    data_desc=f"Cell crops, {len(X_train)} train, {len(X_test)} test",
    labels="normal vs abnormal (binary)",
    split_info="Image-level split, 70/15/15",
    metrics=metrics_rf,
    next_steps="Try XGBoost."
)

# --- Model 3: XGBoost ---
xgb_morph = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=(y_train_binary == 0).sum() / (y_train_binary == 1).sum(),
    random_state=SEED, use_label_encoder=False, eval_metric="logloss"
)
xgb_morph.fit(X_train_sc, y_train_binary)
y_pred_xgb = xgb_morph.predict(X_test_sc)
y_prob_xgb = xgb_morph.predict_proba(X_test_sc)[:, 1]

metrics_xgb = classification_metrics(y_test_binary, y_pred_xgb, y_prob_xgb)
morph_binary_results["XGBoost"] = metrics_xgb
save_checkpoint(xgb_morph, "morph_binary_xgb")

print("\n" + "="*80)
print("BINARY MORPHOLOGY: XGBoost")
print("="*80)
print(classification_report(y_test_binary, y_pred_xgb, target_names=["normal", "abnormal"]))

log_experiment(
    stage="cell_morphology_binary", model_name="XGBoost",
    data_desc=f"Cell crops, {len(X_train)} train, {len(X_test)} test",
    labels="normal vs abnormal (binary)",
    split_info="Image-level split, 70/15/15",
    metrics=metrics_xgb,
)

# --- Comparison ---
print("\n" + "="*80)
print("BINARY MORPHOLOGY MODEL COMPARISON")
print("="*80)
comparison_df = pd.DataFrame(morph_binary_results).T
display(comparison_df.round(3))

In [ ]:
# ── 10C: Multiclass Morphology Classification ────────────────────────────────
# Full multiclass morphology classification using actual data classes.

morph_multi_results = {}

# Dynamically determine classes from the training data
morph_classes = sorted(morph_ml_df["class_name"].unique().tolist())
n_classes = len(morph_classes)
class_to_idx = {c: i for i, c in enumerate(morph_classes)}

# Re-encode targets to ensure contiguous 0..N-1 labels
morph_ml_df["class_idx_remapped"] = morph_ml_df["class_name"].map(class_to_idx)
y_train_multi = morph_ml_df[morph_ml_df["split"] == "train"]["class_idx_remapped"].values
y_val_multi = morph_ml_df[morph_ml_df["split"] == "val"]["class_idx_remapped"].values
y_test_multi = morph_ml_df[morph_ml_df["split"] == "test"]["class_idx_remapped"].values

print(f"Multiclass morphology: {n_classes} classes")
for cls, idx in class_to_idx.items():
    n_train = (y_train_multi == idx).sum()
    n_test = (y_test_multi == idx).sum()
    print(f"  [{idx}] {cls}: train={n_train}, test={n_test}")

# --- Random Forest (multiclass) ---
rf_multi = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                   max_depth=12, random_state=SEED, n_jobs=-1)
rf_multi.fit(X_train_sc, y_train_multi)
y_pred_rf_multi = rf_multi.predict(X_test_sc)

metrics_rf_multi = classification_metrics(y_test_multi, y_pred_rf_multi, average="macro")
morph_multi_results["RandomForest"] = metrics_rf_multi
save_checkpoint(rf_multi, "morph_multi_rf")

print("\n" + "="*80)
print("MULTICLASS MORPHOLOGY: Random Forest")
print("="*80)
print(classification_report(y_test_multi, y_pred_rf_multi, 
                            labels=list(range(n_classes)), target_names=morph_classes, zero_division=0))

# --- XGBoost (multiclass) ---
xgb_multi = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    objective="multi:softprob", num_class=n_classes,
    random_state=SEED, use_label_encoder=False, eval_metric="mlogloss"
)
xgb_multi.fit(X_train_sc, y_train_multi)
y_pred_xgb_multi = xgb_multi.predict(X_test_sc)

metrics_xgb_multi = classification_metrics(y_test_multi, y_pred_xgb_multi, average="macro")
morph_multi_results["XGBoost"] = metrics_xgb_multi
save_checkpoint(xgb_multi, "morph_multi_xgb")

print("\n" + "="*80)
print("MULTICLASS MORPHOLOGY: XGBoost")
print("="*80)
print(classification_report(y_test_multi, y_pred_xgb_multi, 
                            labels=list(range(n_classes)), target_names=morph_classes, zero_division=0))

# --- LightGBM (multiclass) ---
lgb_multi = lgb.LGBMClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    class_weight="balanced", random_state=SEED, verbose=-1
)
lgb_multi.fit(X_train_sc, y_train_multi)
y_pred_lgb_multi = lgb_multi.predict(X_test_sc)

metrics_lgb_multi = classification_metrics(y_test_multi, y_pred_lgb_multi, average="macro")
morph_multi_results["LightGBM"] = metrics_lgb_multi
save_checkpoint(lgb_multi, "morph_multi_lgb")

print("\n" + "="*80)
print("MULTICLASS MORPHOLOGY: LightGBM")
print("="*80)
print(classification_report(y_test_multi, y_pred_lgb_multi, 
                            labels=list(range(n_classes)), target_names=morph_classes, zero_division=0))

# Comparison
print("\n" + "="*80)
print("MULTICLASS MORPHOLOGY MODEL COMPARISON")
print("="*80)
multi_comparison = pd.DataFrame(morph_multi_results).T
display(multi_comparison.round(3))

# Best model confusion matrix
best_model_name = multi_comparison["f1_macro"].idxmax()
best_preds = {"RandomForest": y_pred_rf_multi, "XGBoost": y_pred_xgb_multi, "LightGBM": y_pred_lgb_multi}
plot_confusion(y_test_multi, best_preds[best_model_name], morph_classes,
               title=f"Best Morph Model ({best_model_name}) — Confusion Matrix")
plt.savefig(PATHS.reports / "morph_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

log_experiment(
    stage="cell_morphology_multiclass", model_name=best_model_name,
    data_desc=f"Cell crops ({data_source_type}), {len(X_train)} train, {len(X_test)} test, {n_classes} classes",
    labels=f"{n_classes}-class morphology: {morph_classes}",
    split_info="Image-level split, 70/15/15",
    metrics=morph_multi_results[best_model_name],
)

In [ ]:
# ── 10D: Feature Importance ───────────────────────────────────────────────────

# XGBoost feature importance
importances = pd.Series(xgb_multi.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
importances.head(15).plot.barh(ax=ax, color="steelblue")
ax.set_title("Top 15 Morphology Features (XGBoost Importance)")
ax.set_xlabel("Importance Score")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PATHS.reports / "morph_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top 10 features:")
for feat, imp in importances.head(10).items():
    print(f"  {feat}: {imp:.4f}")

# ── 10E: Error Analysis ──────────────────────────────────────────────────────
# Examine misclassified cells

best_multi_preds = best_preds[best_model_name]
test_morph_analysis = test_morph.copy()
test_morph_analysis["predicted"] = best_multi_preds
test_morph_analysis["correct"] = test_morph_analysis["class_idx"] == test_morph_analysis["predicted"]

error_rate_by_class = test_morph_analysis.groupby("class_name")["correct"].mean()
print("\nAccuracy by class:")
print(error_rate_by_class.round(3).to_string())

# Most confused pairs
cm = confusion_matrix(y_test_multi, best_multi_preds)
np.fill_diagonal(cm, 0)
max_confusion_idx = np.unravel_index(cm.argmax(), cm.shape)
print(f"\nMost confused pair: {morph_classes[max_confusion_idx[0]]} -> {morph_classes[max_confusion_idx[1]]} ({cm[max_confusion_idx]} misclassifications)")

print("\n--- Cell-Level Modeling Complete ---")

---

## Section 11 — Modeling: Track Level

**Purpose:** Classify motility states and extract kinematic features from track-level data.

**Inputs:** CASA kinematic parameters, derived motility labels.

**Outputs:** Track-level classifiers, kinematic predictions.

In [ ]:
# ── 11A: Track-Level Motility Classification ─────────────────────────────────
# Since we don't have real video tracking data, we use the CASA kinematic
# parameters as proxy track features and classify motility state.

# Encode motility labels
motility_le = LabelEncoder()
casa_df["motility_encoded"] = motility_le.fit_transform(casa_df["rule_motility_class"])
motility_class_names = list(motility_le.classes_)

# Track-level features (kinematic parameters)
track_features = ["vcl_um_s", "vsl_um_s", "vap_um_s", "linearity_pct", 
                  "straightness_pct", "wobble_pct", "alh_um", "bcf_hz"]

# Split by bull
train_casa = casa_df[casa_df["split"] == "train"]
val_casa = casa_df[casa_df["split"] == "val"]
test_casa = casa_df[casa_df["split"] == "test"]

X_train_track = train_casa[track_features].fillna(0).values
y_train_track = train_casa["motility_encoded"].values
X_val_track = val_casa[track_features].fillna(0).values
y_val_track = val_casa["motility_encoded"].values
X_test_track = test_casa[track_features].fillna(0).values
y_test_track = test_casa["motility_encoded"].values

scaler_track = RobustScaler()
X_train_track_sc = scaler_track.fit_transform(X_train_track)
X_val_track_sc = scaler_track.transform(X_val_track)
X_test_track_sc = scaler_track.transform(X_test_track)

print(f"Track-level data (motility classification):")
print(f"  Train: {len(X_train_track)} samples")
print(f"  Val:   {len(X_val_track)} samples")
print(f"  Test:  {len(X_test_track)} samples")
print(f"  Classes: {motility_class_names}")
print(f"  Train distribution: {dict(zip(*np.unique(y_train_track, return_counts=True)))}")

track_results = {}

# --- GradientBoosting ---
gb_track = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1, random_state=SEED
)
gb_track.fit(X_train_track_sc, y_train_track)
y_pred_gb_track = gb_track.predict(X_test_track_sc)

metrics_gb_track = classification_metrics(y_test_track, y_pred_gb_track, average="macro")
track_results["GradientBoosting"] = metrics_gb_track
save_checkpoint(gb_track, "track_motility_gb")

print("\n" + "="*80)
print("TRACK MOTILITY: Gradient Boosting")
print("="*80)
print(classification_report(y_test_track, y_pred_gb_track, 
                            target_names=motility_class_names, zero_division=0))

# --- LightGBM ---
lgb_track = lgb.LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    class_weight="balanced", random_state=SEED, verbose=-1
)
lgb_track.fit(X_train_track_sc, y_train_track)
y_pred_lgb_track = lgb_track.predict(X_test_track_sc)

metrics_lgb_track = classification_metrics(y_test_track, y_pred_lgb_track, average="macro")
track_results["LightGBM"] = metrics_lgb_track
save_checkpoint(lgb_track, "track_motility_lgb")

print("\n" + "="*80)
print("TRACK MOTILITY: LightGBM")
print("="*80)
print(classification_report(y_test_track, y_pred_lgb_track,
                            target_names=motility_class_names, zero_division=0))

# Comparison
print("\n" + "="*80)
print("TRACK MOTILITY MODEL COMPARISON")
print("="*80)
track_comparison = pd.DataFrame(track_results).T
display(track_comparison.round(3))

# Confusion matrix
best_track = track_comparison["f1_macro"].idxmax()
best_track_preds = y_pred_gb_track if best_track == "GradientBoosting" else y_pred_lgb_track
plot_confusion(y_test_track, best_track_preds, motility_class_names,
               title=f"Track Motility ({best_track}) — Confusion Matrix")
plt.savefig(PATHS.reports / "track_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

log_experiment(
    stage="track_motility", model_name=best_track,
    data_desc=f"CASA kinematic features, {len(X_train_track)} train, {len(X_test_track)} test",
    labels="immotile / non-progressive / progressive (derived from rule-based CASA thresholds)",
    split_info="Bull-level split, 70/15/15",
    metrics=track_results[best_track],
    notes="Labels are derived from rule-based classification, not expert annotation.",
)

print("\n--- Track-Level Modeling Complete ---")

---

## Section 12 — Modeling: Sample Level

**Purpose:** Predict sample-level quality metrics: concentration, total motility, progressive motility, morphology %, quality class, and fertility proxy.

**Models:** Linear -> ElasticNet -> Random Forest -> XGBoost -> LightGBM

**Inputs:** Engineered sample features, CASA data, bull-level splits.

**Outputs:** Regression/classification predictions, model comparison tables, calibration analysis, SHAP-like interpretation.

In [ ]:
# ── 12A: Sample-Level Feature Preparation ─────────────────────────────────────

# Sample-level features for modeling
sample_feature_cols = [
    # CASA metrics
    "concentration_M_per_mL", "total_motility_pct", "progressive_motility_pct",
    "morphology_normal_pct", "vcl_um_s", "vsl_um_s", "vap_um_s",
    "linearity_pct", "straightness_pct", "wobble_pct", "alh_um", "bcf_hz",
    "head_defects_pct", "midpiece_defects_pct", "tail_defects_pct", "droplets_pct",
    # Engineered features
    "fe_is_summer", "fe_is_winter", "fe_is_frozen",
    "fe_prog_to_total_ratio", "fe_immotile_pct",
    "fe_vcl_vsl_ratio", "fe_speed_index", "fe_total_defects_pct",
    "fe_motility_morph_score", "fe_log_concentration_M_per_mL", "fe_log_vcl_um_s",
    "fe_motility_vs_bull_mean", "fe_morphology_vs_bull_mean",
    "bull_age_years",
]

# Only use features that exist
sample_feature_cols = [c for c in sample_feature_cols if c in casa_df.columns]

# Split data
train_s = casa_df[casa_df["split"] == "train"].copy()
val_s = casa_df[casa_df["split"] == "val"].copy()
test_s = casa_df[casa_df["split"] == "test"].copy()

print(f"Sample-level splits (by bull):")
print(f"  Train: {len(train_s)} samples ({train_s['bull_id'].nunique()} bulls)")
print(f"  Val:   {len(val_s)} samples ({val_s['bull_id'].nunique()} bulls)")
print(f"  Test:  {len(test_s)} samples ({test_s['bull_id'].nunique()} bulls)")
print(f"  Features: {len(sample_feature_cols)}")

In [ ]:
# ── 12B: Sample Quality Classification (pass/review/reject) ──────────────────

quality_le = LabelEncoder()
casa_df["quality_encoded"] = quality_le.fit_transform(casa_df["quality_class"])
quality_names = list(quality_le.classes_)

# Re-extract splits after encoding
train_s = casa_df[casa_df["split"] == "train"].copy()
val_s = casa_df[casa_df["split"] == "val"].copy()
test_s = casa_df[casa_df["split"] == "test"].copy()

# Use features that don't directly leak the target
# Remove motility and morphology raw values for quality prediction (they define the target)
quality_feature_cols = [c for c in sample_feature_cols if c not in 
                        ["total_motility_pct", "progressive_motility_pct", "morphology_normal_pct",
                         "concentration_M_per_mL"]]

X_train_q = train_s[quality_feature_cols].fillna(0).values
y_train_q = train_s["quality_encoded"].values
X_val_q = val_s[quality_feature_cols].fillna(0).values
y_val_q = val_s["quality_encoded"].values
X_test_q = test_s[quality_feature_cols].fillna(0).values
y_test_q = test_s["quality_encoded"].values

scaler_q = RobustScaler()
X_train_q_sc = scaler_q.fit_transform(X_train_q)
X_val_q_sc = scaler_q.transform(X_val_q)
X_test_q_sc = scaler_q.transform(X_test_q)

quality_results = {}

# --- Logistic Regression ---
lr_q = LogisticRegression(max_iter=2000, class_weight="balanced", multi_class="multinomial", random_state=SEED)
lr_q.fit(X_train_q_sc, y_train_q)
y_pred_lr_q = lr_q.predict(X_test_q_sc)
metrics_lr_q = classification_metrics(y_test_q, y_pred_lr_q, average="macro")
quality_results["LogisticRegression"] = metrics_lr_q

# --- Random Forest ---
rf_q = RandomForestClassifier(n_estimators=300, class_weight="balanced", max_depth=8, 
                               random_state=SEED, n_jobs=-1)
rf_q.fit(X_train_q_sc, y_train_q)
y_pred_rf_q = rf_q.predict(X_test_q_sc)
metrics_rf_q = classification_metrics(y_test_q, y_pred_rf_q, average="macro")
quality_results["RandomForest"] = metrics_rf_q

# --- XGBoost ---
xgb_q = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.1,
    objective="multi:softprob", num_class=len(quality_names),
    random_state=SEED, use_label_encoder=False, eval_metric="mlogloss"
)
xgb_q.fit(X_train_q_sc, y_train_q)
y_pred_xgb_q = xgb_q.predict(X_test_q_sc)
metrics_xgb_q = classification_metrics(y_test_q, y_pred_xgb_q, average="macro")
quality_results["XGBoost"] = metrics_xgb_q

# --- LightGBM ---
lgb_q = lgb.LGBMClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    class_weight="balanced", random_state=SEED, verbose=-1
)
lgb_q.fit(X_train_q_sc, y_train_q)
y_pred_lgb_q = lgb_q.predict(X_test_q_sc)
metrics_lgb_q = classification_metrics(y_test_q, y_pred_lgb_q, average="macro")
quality_results["LightGBM"] = metrics_lgb_q

# Comparison
print("="*80)
print("SAMPLE QUALITY CLASSIFICATION (pass/review/reject)")
print("="*80)
q_comparison = pd.DataFrame(quality_results).T
display(q_comparison.round(3))

# Best model
best_q = q_comparison["f1_macro"].idxmax()
best_q_preds = {"LogisticRegression": y_pred_lr_q, "RandomForest": y_pred_rf_q,
                "XGBoost": y_pred_xgb_q, "LightGBM": y_pred_lgb_q}[best_q]
save_checkpoint({"LogisticRegression": lr_q, "RandomForest": rf_q, 
                 "XGBoost": xgb_q, "LightGBM": lgb_q}[best_q], "sample_quality_best")

print(f"\nBest model: {best_q}")
print(classification_report(y_test_q, best_q_preds, target_names=quality_names, zero_division=0))

plot_confusion(y_test_q, best_q_preds, quality_names,
               title=f"Sample Quality ({best_q}) — Confusion Matrix")
plt.savefig(PATHS.reports / "sample_quality_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

log_experiment(
    stage="sample_quality_classification", model_name=best_q,
    data_desc=f"CASA + engineered features, {len(X_train_q)} train, {len(X_test_q)} test",
    labels="pass/review/reject",
    split_info="Bull-level split, 70/15/15",
    metrics=quality_results[best_q],
    notes="Features exclude raw motility/morphology targets to avoid trivial leakage.",
)

In [ ]:
# ── 12C: Sample-Level Regression (Fertility Proxy) ────────────────────────────
# Predict fertility_proxy from kinematic + defect + metadata features.
# Only use samples where fertility_proxy is available.

fert_df = casa_df.dropna(subset=["fertility_proxy"]).copy()

# Re-split — ensure bull-level splits are preserved
train_f = fert_df[fert_df["split"] == "train"]
val_f = fert_df[fert_df["split"] == "val"]  
test_f = fert_df[fert_df["split"] == "test"]

# Regression features (all sample features)
reg_feature_cols = [c for c in sample_feature_cols if c != "fertility_proxy" and c in fert_df.columns]

X_train_f = train_f[reg_feature_cols].fillna(0).values
y_train_f = train_f["fertility_proxy"].values
X_val_f = val_f[reg_feature_cols].fillna(0).values
y_val_f = val_f["fertility_proxy"].values
X_test_f = test_f[reg_feature_cols].fillna(0).values
y_test_f = test_f["fertility_proxy"].values

scaler_f = RobustScaler()
X_train_f_sc = scaler_f.fit_transform(X_train_f)
X_val_f_sc = scaler_f.transform(X_val_f)
X_test_f_sc = scaler_f.transform(X_test_f)

print(f"Fertility regression data:")
print(f"  Train: {len(X_train_f)} samples")
print(f"  Val:   {len(X_val_f)} samples")  
print(f"  Test:  {len(X_test_f)} samples")
print(f"  Target range: {y_train_f.min():.1f} - {y_train_f.max():.1f}")

fertility_results = {}

# --- Linear Regression ---
from sklearn.linear_model import LinearRegression
lr_f = LinearRegression()
lr_f.fit(X_train_f_sc, y_train_f)
y_pred_lr_f = lr_f.predict(X_test_f_sc)
fertility_results["LinearRegression"] = regression_metrics(y_test_f, y_pred_lr_f)

# --- ElasticNet ---
en_f = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000, random_state=SEED)
en_f.fit(X_train_f_sc, y_train_f)
y_pred_en_f = en_f.predict(X_test_f_sc)
fertility_results["ElasticNet"] = regression_metrics(y_test_f, y_pred_en_f)

# --- Random Forest Regressor ---
rf_f = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1)
rf_f.fit(X_train_f_sc, y_train_f)
y_pred_rf_f = rf_f.predict(X_test_f_sc)
fertility_results["RandomForest"] = regression_metrics(y_test_f, y_pred_rf_f)

# --- XGBoost Regressor ---
xgb_f = xgb.XGBRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.1,
    random_state=SEED, eval_metric="rmse"
)
xgb_f.fit(X_train_f_sc, y_train_f)
y_pred_xgb_f = xgb_f.predict(X_test_f_sc)
fertility_results["XGBoost"] = regression_metrics(y_test_f, y_pred_xgb_f)

# --- LightGBM Regressor ---
lgb_f = lgb.LGBMRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    random_state=SEED, verbose=-1
)
lgb_f.fit(X_train_f_sc, y_train_f)
y_pred_lgb_f = lgb_f.predict(X_test_f_sc)
fertility_results["LightGBM"] = regression_metrics(y_test_f, y_pred_lgb_f)

# Comparison
print("\n" + "="*80)
print("FERTILITY PROXY REGRESSION MODEL COMPARISON")
print("="*80)
fert_comparison = pd.DataFrame(fertility_results).T
display(fert_comparison.round(3))

# Best model scatter plot
best_f = fert_comparison["r2"].idxmax()
best_f_preds = {"LinearRegression": y_pred_lr_f, "ElasticNet": y_pred_en_f,
                "RandomForest": y_pred_rf_f, "XGBoost": y_pred_xgb_f, "LightGBM": y_pred_lgb_f}[best_f]
save_checkpoint({"LinearRegression": lr_f, "ElasticNet": en_f, "RandomForest": rf_f,
                 "XGBoost": xgb_f, "LightGBM": lgb_f}[best_f], "fertility_best")

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test_f, best_f_preds, alpha=0.6, edgecolors="navy", s=50)
lims = [min(y_test_f.min(), best_f_preds.min()) - 5, max(y_test_f.max(), best_f_preds.max()) + 5]
ax.plot(lims, lims, "r--", linewidth=2, label="Perfect prediction")
ax.set_xlabel("Actual Fertility Proxy")
ax.set_ylabel("Predicted Fertility Proxy")
ax.set_title(f"Fertility Prediction ({best_f}) — R²={fert_comparison.loc[best_f, 'r2']:.3f}")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(PATHS.reports / "fertility_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

log_experiment(
    stage="sample_fertility_regression", model_name=best_f,
    data_desc=f"CASA features, {len(X_train_f)} train, {len(X_test_f)} test",
    labels="fertility_proxy (continuous, synthetic)",
    split_info="Bull-level split",
    metrics=fertility_results[best_f],
    notes="Fertility proxy is synthetically generated — results show pipeline capability only.",
    next_steps="Replace with real fertility data from AI station records."
)

print("\n--- Sample-Level Modeling Complete ---")

---

## Section 13 — Synthetic Data Experiments

**Purpose:** Evaluate the impact of synthetic data augmentation on model performance.

**Experiments:** Real-only vs Real+Augmented baselines. Full ablation table.

In [ ]:
# ── 13A: Augmentation Impact Experiment ───────────────────────────────────────
# Compare model performance with different data augmentation strategies.
# Since ALL our current data is synthetic, this experiment compares:
#   - Base dataset (300 images)
#   - Base + classical augmentation (doubled via flips/rotation)
# When real data is available, this section should compare:
#   - Real only
#   - Real + augmentation
#   - Real + rule-based synthetic
#   - Real + learned synthetic

def apply_augmentation(img, aug_type="flip_h"):
    """Apply a single augmentation to an image."""
    if aug_type == "flip_h":
        return cv2.flip(img, 1)
    elif aug_type == "flip_v":
        return cv2.flip(img, 0)
    elif aug_type == "rotate_90":
        return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    elif aug_type == "blur":
        return cv2.GaussianBlur(img, (5, 5), 0)
    elif aug_type == "noise":
        noise = np.random.normal(0, 10, img.shape).astype(np.int16)
        return np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    elif aug_type == "brightness":
        factor = np.random.uniform(0.7, 1.3)
        return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)
    return img

# Augmentation experiment: train with more augmented samples
# Create augmented features by adding noise to existing features
np.random.seed(SEED)

train_orig = morph_ml_df[morph_ml_df["split"] == "train"][feature_cols].fillna(0).values
y_train_orig = morph_ml_df[morph_ml_df["split"] == "train"]["binary_target"].values

# Augment by adding small Gaussian noise to features (simulates augmentation)
n_aug = len(train_orig)
noise_scale = 0.05
X_aug = train_orig + np.random.normal(0, noise_scale * train_orig.std(axis=0), train_orig.shape)
y_aug = y_train_orig.copy()

# Combined
X_train_combined = np.vstack([train_orig, X_aug])
y_train_combined = np.concatenate([y_train_orig, y_aug])

# Train models with augmented data
scaler_aug = RobustScaler()
X_train_comb_sc = scaler_aug.fit_transform(X_train_combined)
X_test_aug_sc = scaler_aug.transform(X_test)

ablation_results = {}

# Base (no augmentation)
rf_base = RandomForestClassifier(n_estimators=200, class_weight="balanced", max_depth=10, random_state=SEED)
rf_base.fit(scaler_morph.transform(train_orig), y_train_orig)
y_pred_base = rf_base.predict(X_test_sc)
ablation_results["real_only"] = classification_metrics(y_test_binary, y_pred_base)

# With augmentation
rf_aug = RandomForestClassifier(n_estimators=200, class_weight="balanced", max_depth=10, random_state=SEED)
rf_aug.fit(X_train_comb_sc, y_train_combined)
y_pred_aug = rf_aug.predict(X_test_aug_sc)
ablation_results["real_plus_augmentation"] = classification_metrics(y_test_binary, y_pred_aug)

# Display ablation table
print("="*80)
print("SYNTHETIC DATA ABLATION TABLE")
print("="*80)
ablation_df = pd.DataFrame(ablation_results).T
display(ablation_df.round(3))

# Impact analysis
for metric in ["accuracy", "f1_macro"]:
    base_val = ablation_results["real_only"].get(metric, 0)
    aug_val = ablation_results["real_plus_augmentation"].get(metric, 0)
    delta = aug_val - base_val
    print(f"  {metric}: base={base_val:.3f}, augmented={aug_val:.3f}, delta={delta:+.3f}")

print("\nNOTE: All data in this experiment is synthetic.")
print("With real data, this section would compare:")
print("  1. Real only")
print("  2. Real + classical augmentation")
print("  3. Real + rule-based synthetic")
print("  4. Real + learned synthetic (GAN/diffusion)")

log_experiment(
    stage="synthetic_ablation", model_name="RandomForest",
    data_desc="Synthetic crops: base vs augmented",
    labels="normal vs abnormal (binary)",
    split_info="Image-level split",
    metrics={"base_f1": ablation_results["real_only"].get("f1_macro", 0),
             "augmented_f1": ablation_results["real_plus_augmentation"].get("f1_macro", 0)},
    notes="All data is synthetic. This is a pipeline demonstration.",
)

---

## Section 14 — Results Summary

**Purpose:** Consolidate all modeling results, compare approaches, identify limitations, and determine what generalizes vs what is overfitting.

In [ ]:
# ── 14A: Full Experiment Log ──────────────────────────────────────────────────

print("="*80)
print("COMPLETE EXPERIMENT LOG")
print("="*80)
exp_log_df = show_experiment_log()

# Save experiment log
if len(exp_log_df) > 0:
    exp_log_df.to_csv(PATHS.reports / "experiment_log.csv", index=False)
    print(f"\nExperiment log saved: {len(exp_log_df)} entries")

In [ ]:
# ── 14B: Model Comparison Dashboard ──────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Binary morphology comparison
if morph_binary_results:
    ax = axes[0, 0]
    binary_df = pd.DataFrame(morph_binary_results)
    binary_df.loc[["accuracy", "f1_macro"]].T.plot.bar(ax=ax, color=["steelblue", "coral"])
    ax.set_title("Binary Morphology (Normal vs Abnormal)")
    ax.set_ylabel("Score")
    ax.legend(loc="lower right")
    ax.set_ylim(0, 1)

# 2. Multiclass morphology comparison
if morph_multi_results:
    ax = axes[0, 1]
    multi_df = pd.DataFrame(morph_multi_results)
    multi_df.loc[["accuracy", "f1_macro"]].T.plot.bar(ax=ax, color=["steelblue", "coral"])
    ax.set_title("Multiclass Morphology (6 classes)")
    ax.set_ylabel("Score")
    ax.legend(loc="lower right")
    ax.set_ylim(0, 1)

# 3. Track motility comparison
if track_results:
    ax = axes[1, 0]
    track_df = pd.DataFrame(track_results)
    track_df.loc[["accuracy", "f1_macro"]].T.plot.bar(ax=ax, color=["steelblue", "coral"])
    ax.set_title("Track Motility Classification")
    ax.set_ylabel("Score")
    ax.legend(loc="lower right")
    ax.set_ylim(0, 1)

# 4. Sample quality + fertility
ax = axes[1, 1]
if quality_results:
    q_df = pd.DataFrame(quality_results)
    q_df.loc[["f1_macro"]].T.plot.bar(ax=ax, color=["teal"])
    ax.set_title("Sample Quality Classification (F1 Macro)")
    ax.set_ylabel("F1 Score")
    ax.set_ylim(0, 1)

plt.suptitle("Model Performance Dashboard — All Levels", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(PATHS.reports / "model_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

# ── 14C: Feature Importance Summary ──────────────────────────────────────────
# Show feature importance for sample-level quality model
if hasattr(xgb_q, "feature_importances_"):
    q_importance = pd.Series(xgb_q.feature_importances_, index=quality_feature_cols).sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    q_importance.head(15).plot.barh(ax=ax, color="teal")
    ax.set_title("Top 15 Sample Quality Prediction Features")
    ax.set_xlabel("Importance")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(PATHS.reports / "quality_feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── 14D: Sample-Level Report Generator ────────────────────────────────────────
# Generate a compact quality report for each test sample.

def generate_sample_report(row, quality_model, quality_scaler, feature_cols, 
                           quality_le, fertility_model=None, fertility_scaler=None, 
                           fert_feature_cols=None):
    """Generate a quality report for a single sample."""
    # Quality prediction
    X = np.array([row[feature_cols].fillna(0).values])
    X_sc = quality_scaler.transform(X)
    q_pred = quality_model.predict(X_sc)[0]
    q_proba = quality_model.predict_proba(X_sc)[0]
    q_class = quality_le.inverse_transform([q_pred])[0]
    
    # Fertility prediction if model exists
    fert_pred = None
    if fertility_model and fert_feature_cols:
        X_f = np.array([row[fert_feature_cols].fillna(0).values])
        X_f_sc = fertility_scaler.transform(X_f)
        fert_pred = fertility_model.predict(X_f_sc)[0]
    
    report = {
        "sample_id": row.get("sample_id", "unknown"),
        "bull_id": row.get("bull_id", "unknown"),
        "predicted_quality": q_class,
        "quality_confidence": float(max(q_proba)),
        "quality_probabilities": {quality_le.inverse_transform([i])[0]: float(p) 
                                   for i, p in enumerate(q_proba)},
        "predicted_fertility": round(fert_pred, 1) if fert_pred else None,
        "total_motility_pct": row.get("total_motility_pct"),
        "progressive_motility_pct": row.get("progressive_motility_pct"),
        "morphology_normal_pct": row.get("morphology_normal_pct"),
        "concentration_M_per_mL": row.get("concentration_M_per_mL"),
        "top_defect": row.get("fe_major_defect_type", "unknown"),
    }
    return report

# Generate reports for test samples
test_reports = []
best_q_model = {"LogisticRegression": lr_q, "RandomForest": rf_q, 
                "XGBoost": xgb_q, "LightGBM": lgb_q}[best_q]

for _, row in test_s.head(10).iterrows():
    report = generate_sample_report(
        row, best_q_model, scaler_q, quality_feature_cols, quality_le,
        fertility_model=lgb_f if len(X_test_f) > 0 else None,
        fertility_scaler=scaler_f if len(X_test_f) > 0 else None,
        fert_feature_cols=reg_feature_cols if len(X_test_f) > 0 else None,
    )
    test_reports.append(report)

print("="*80)
print("SAMPLE QUALITY REPORTS (Test Set — First 10)")
print("="*80)
for r in test_reports:
    print(f"\n  Sample: {r['sample_id']} | Bull: {r['bull_id']}")
    print(f"  Quality: {r['predicted_quality']} (confidence: {r['quality_confidence']:.2f})")
    print(f"  Motility: {r['total_motility_pct']}% total, {r['progressive_motility_pct']}% progressive")
    print(f"  Morphology: {r['morphology_normal_pct']}% normal")
    if r["predicted_fertility"]:
        print(f"  Fertility proxy: {r['predicted_fertility']}")
    print(f"  Top defect: {r['top_defect']}")

# Save predictions
predictions_df = pd.DataFrame(test_reports)
save_manifest(predictions_df, PATHS.outputs / "sample_predictions.csv")

---

## Section 15 — Next Steps

### Executive Summary

This notebook demonstrates a complete, end-to-end prototype pipeline for **bovine sperm quality analysis** using synthetic data as a stand-in for real bovine microscopy and CASA data. The pipeline covers:

1. **Data discovery** — 10 candidate sources cataloged across Roboflow, Figshare, VISEM, Kaggle, Zenodo
2. **Data acquisition** — Automated download pipeline with integrity checks and manifests
3. **Data audit** — Comprehensive QC with corruption checks, class balance analysis, and split unit determination
4. **Label taxonomy** — Hierarchical labels for morphology (7 classes), motion (4 classes), and sample quality
5. **Data preparation** — Crop extraction, preprocessing, leakage-safe splitting by bull/image
6. **Feature engineering** — 11+ classical morphology features, 15+ engineered sample features
7. **Modeling at 3 levels:**
   - **Cell-level:** Binary and 6-class morphology classification (LogReg, RF, XGBoost, LightGBM)
   - **Track-level:** Motility classification from kinematic features (GradientBoosting, LightGBM)
   - **Sample-level:** Quality classification and fertility regression (5 models each)
8. **Synthetic data experiments** — Ablation comparing base vs augmented
9. **Report generation** — Per-sample quality reports with predictions

### What is based on public data
- Source catalog from systematic web search
- Pipeline structure inspired by OpenCASA and published bovine reproduction literature
- Feature engineering based on standard CASA kinematic parameters

### What is synthetic / inferred
- ALL images and tabular data in this prototype are synthetically generated
- Model performance reflects pipeline functionality, NOT real-world bovine sperm analysis accuracy
- Synthetic data was designed to have biologically plausible distributions but is NOT validated

### What would require private lab data
- Real microscopy images of bovine sperm (from AI stations)
- Expert-annotated morphology labels
- Actual CASA exports with calibrated kinematic parameters
- Bull fertility records (conception rates, non-return rates)
- Post-thaw quality measurements
- Device/operator metadata
- Multi-lab validation data for domain robustness

### Data gaps
- No real bovine morphology images downloaded (Roboflow requires API key)
- No real CASA kinematic data
- No fertility outcome data
- No video data for actual tracking
- No pixel-to-micron calibration

### Recommended next steps
1. **Obtain Roboflow API key** and download real bovine sperm morphology datasets
2. **Partner with AI station** to get CASA exports, fertility records, and microscopy images
3. **Run expert annotation** on a subset of real images for gold-standard labels
4. **Replace synthetic data** throughout and re-run the full pipeline
5. **Add real video processing** for track-level features
6. **Implement CNN/YOLOv8** detection and classification with real data
7. **Validate across labs** to test domain robustness
8. **Calibrate predictions** for deployment confidence

### Ready for prototyping
- Complete end-to-end pipeline structure
- All helper functions, schemas, and manifests
- Model training and evaluation framework
- Report generation system

### Not yet trustworthy for production
- No real data validation
- No expert label quality assessment
- No cross-lab domain testing
- No temporal stability testing
- No regulatory/compliance review

In [ ]:
# ── Final Output Summary ──────────────────────────────────────────────────────

print("="*80)
print("NOTEBOOK EXECUTION COMPLETE")
print("="*80)

# List all output files
output_files = []
for d in [PATHS.manifests, PATHS.processed, PATHS.outputs, PATHS.reports, 
          PATHS.models, PATHS.configs]:
    for f in sorted(d.rglob("*")):
        if f.is_file():
            output_files.append({"path": str(f.relative_to(PATHS.root)), 
                                 "size_kb": f.stat().st_size / 1024})

output_df = pd.DataFrame(output_files)
if len(output_df) > 0:
    print(f"\nGenerated {len(output_df)} output files:")
    print(f"  Total size: {output_df['size_kb'].sum():.1f} KB")
    print(f"\n  Key outputs:")
    for _, row in output_df.iterrows():
        print(f"    {row['path']} ({row['size_kb']:.1f} KB)")

print(f"\n  Models saved: {len(list(PATHS.models.glob('*.pkl')))}")
print(f"  Manifests: {len(list(PATHS.manifests.glob('*.csv')))}")
print(f"  Reports: {len(list(PATHS.reports.glob('*')))}")

print("\n" + "="*80)
print("Pipeline ready for real data. Set ROBOFLOW_API_KEY and re-run.")
print("="*80)